# 3. Evaluate

Produces every number and figure of the paper. It needs the benchmark from notebook 1 and checkpoints from notebook 2.

| Section | Output |
|---|---|
| Folding check | A velocity network at t = 0 equals a `DirectMapping`: identical outputs |
| COCO benchmark | Main table, share of the achievable gain, gray/colour split, direct vs flow ablation |
| Efficiency | Parameters, GFLOPs, ms/img (mapping and full pipeline, batch 1 and 16) |
| Colour | CIELAB analysis on the 500-pair held-out set |
| Real photos | MUSIQ, CLIP-IQA, NIQE on real old photographs |
| Figures | `qualitative.pdf`, `training_curve.pdf` |

All outputs are evaluated **raw**, with no post-processing. Results are cached in `results/coco_all.csv`; re-running scores only what is new.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
if not os.path.isdir("/content/Bringing-Old-Photos-Back-to-Life"):
    subprocess.run(["git", "clone", "-q", "https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life",
                    "/content/Bringing-Old-Photos-Back-to-Life"], check=True)
sys.path.insert(0, "/content")
!pip install -q lpips pytorch-fid pyiqa

## Code (identical to `src/`)

### `old_photo_model.py`

In [ ]:
%%writefile /content/old_photo_model.py
"""
OldPhotoModel: single-class wrapper around microsoft/Bringing-Old-Photos-Back-to-Life
(Global restoration stage). Finds the repo (Colab: /content/...), downloads the pretrained
weights if missing, builds VAE A, VAE B and the mapping network, loads their weights and
verifies that every checkpoint key matched.

    from old_photo_model import OldPhotoModel
    obj = OldPhotoModel(mode="quality", gpu=0)

    x   = obj.to_tensor(PIL_image)
    z_A = obj.encode_A(x)
    z_B = obj.map(z_A)
    y   = obj.decode_B(z_B)            # or obj.translate(x)
    obj.to_image(y).save("out.png")
    obj.restore(PIL_image).save("out.png")

Modes: "quality", "scratch" (needs mask), "scratch_hr" (needs mask, high-res).
Scratch modes: x, m = obj.prepare(image, mask); obj.map(z_A, m) / obj.translate(x, m).
"""
import contextlib
import glob
import os
import shutil
import sys
import tempfile
import urllib.request
import zipfile

import numpy as np
import torch
from PIL import Image, ImageFilter

_REPO_ZIP = "https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/archive/refs/heads/master.zip"
_CKPT_ZIP = ("https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/"
             "releases/download/v1.0/global_checkpoints.zip")
_DEFAULT_ROOT = "/content/Bringing-Old-Photos-Back-to-Life"

# (checkpoint sub-folder, network label) required per mode -> files named "<epoch>_net_<label>.pth"
_REQUIRED = {
    "quality":    [("VAE_A_quality", "G"), ("VAE_B_quality", "G"), ("mapping_quality", "mapping_net")],
    "scratch":    [("VAE_A_quality", "G"), ("VAE_B_scratch", "G"), ("mapping_scratch", "mapping_net")],
    "scratch_hr": [("VAE_A_quality", "G"), ("VAE_B_scratch", "G"), ("mapping_Patch_Attention", "mapping_net")],
}


def _download(url, dest):
    print(f"[OldPhotoModel] downloading {url}")
    with urllib.request.urlopen(url) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)


def _find_repo(hint=None):
    """Return repo root: hint, /content/<repo>, or any folder under /content containing Global/models/mapping_model.py."""
    marker = os.path.join("Global", "models", "mapping_model.py")
    cands = [hint, _DEFAULT_ROOT, _DEFAULT_ROOT + "-master"]
    cands += glob.glob("/content/*/") + glob.glob("/content/*/*/")
    for c in cands:
        if c and os.path.isfile(os.path.join(c, marker)):
            return os.path.abspath(c)
    return None


@contextlib.contextmanager
def _cpu_safe_torch_load():
    """The repo calls torch.load(path) without map_location, which fails on CPU-only machines."""
    orig = torch.load

    def patched(*args, **kwargs):
        kwargs.setdefault("map_location", "cpu")
        return orig(*args, **kwargs)

    torch.load = patched
    try:
        yield
    finally:
        torch.load = orig


class OldPhotoModel:

    def __init__(self, repo_root=None, mode="quality", gpu=0, auto_download=True, verify=True,
                 test_mode="Full"):
        assert mode in _REQUIRED, f"mode must be one of {list(_REQUIRED)}"
        assert test_mode in ("Full", "Scale"), "test_mode: 'Full' (keep size) or 'Scale' (short side -> 256)"
        self.mode = mode
        self.test_mode = test_mode  # only used in quality mode, like --test_mode in Global/test.py

        found = _find_repo(repo_root)
        if found is None:
            if not auto_download:
                raise FileNotFoundError("Repo not found under /content; pass repo_root=...")
            repo_root = os.path.abspath(repo_root or _DEFAULT_ROOT)
            self._ensure_repo(repo_root)  # zip download as fallback
        else:
            repo_root = found
        global_dir = os.path.join(repo_root, "Global")
        print(f"[OldPhotoModel] using repo at {repo_root}")

        if auto_download:
            self._ensure_weights(global_dir, mode)
        else:
            self._check_weights(global_dir, mode, raise_error=True)

        if gpu >= 0 and not torch.cuda.is_available():
            print("[OldPhotoModel] CUDA not available -> running on CPU")
            gpu = -1

        opt = self._build_opt(global_dir, mode, gpu)

        from models.mapping_model import Pix2PixHDModel_Mapping  # repo code

        prev = os.getcwd()
        os.chdir(global_dir)
        try:
            with _cpu_safe_torch_load():
                core = Pix2PixHDModel_Mapping()
                core.initialize(opt)  # builds netG_A, netG_B, mapping_net + loads weights
        finally:
            os.chdir(prev)

        self.vae_A = core.netG_A      # old-photo domain VAE
        self.vae_B = core.netG_B      # clean domain VAE
        self.mapping = core.mapping_net
        for net in (self.vae_A, self.vae_B, self.mapping):
            net.eval()
            for p in net.parameters():
                p.requires_grad = False

        self.uses_mask = bool(getattr(opt, "NL_use_mask", False))
        self._hr = bool(getattr(opt, "inference_optimize", False))
        self._mask_dilation = int(getattr(opt, "mask_dilation", 0) or 0)
        self.device = next(self.mapping.parameters()).device
        self._core = core
        self._opt = opt

        if verify:
            self._verify_weights(opt, mode)

    # ------------------------------------------------------------------ #
    # Core methods
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def encode_A(self, x):
        return self.vae_A.forward(x, flow="enc")

    @torch.no_grad()
    def encode_B(self, x):
        return self.vae_B.forward(x, flow="enc")

    @torch.no_grad()
    def map(self, z_A, mask=None):
        if not self.uses_mask:
            return self.mapping(z_A)
        if mask is None:
            raise ValueError("scratch mode: pass mask (1,1,H,W) tensor, same H/W as the input image "
                             "(use obj.prepare(image, mask)).")
        if self._hr:
            return self.mapping.inference_forward(z_A, mask)
        return self.mapping(z_A, mask)

    @torch.no_grad()
    def decode_A(self, z_A):
        return self.vae_A.forward(z_A, flow="dec")

    @torch.no_grad()
    def decode_B(self, z_B):
        return self.vae_B.forward(z_B, flow="dec")

    @torch.no_grad()
    def translate(self, x_A, mask=None):
        z_A = self.encode_A(x_A)
        z_B = self.map(z_A, mask)
        return self.decode_B(z_B)

    # ------------------------------------------------------------------ #
    # Helpers: PIL <-> tensor, and one-call restore
    # ------------------------------------------------------------------ #
    def _fit(self, img):
        """Same resizing as Global/test.py (BILINEAR, multiples of 4).
        Scratch modes never rescale in test.py, so only the multiple-of-4 rounding applies there."""
        w, h = img.size
        ow, oh = w, h
        if self.test_mode == "Scale" and not self.uses_mask:
            if w < h:
                ow, oh = 256, h / w * 256
            else:
                oh, ow = 256, w / h * 256
        size = (int(round(ow / 4) * 4), int(round(oh / 4) * 4))
        return img if size == (w, h) else img.resize(size, Image.BILINEAR)

    def to_tensor(self, image):
        """PIL -> (1,3,H,W) tensor in [-1,1]; H,W rounded to multiples of 4."""
        image = self._fit(image.convert("RGB"))
        x = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return ((x - 0.5) / 0.5).unsqueeze(0).to(self.device)

    def mask_to_tensor(self, mask, size):
        """PIL mask (white = scratch) -> (1,1,H,W) tensor, resized to size=(W,H)."""
        m = mask.convert("RGB").resize(size, Image.NEAREST)
        m = torch.from_numpy(np.asarray(m, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return m[:1].unsqueeze(0).to(self.device)

    def prepare(self, image, mask=None):
        """PIL (+ PIL mask in scratch modes) -> (x, mask_tensor or None), preprocessed like Global/test.py:
        size handled like test.py, mask dilated (HR mode), scratched pixels painted white."""
        img = self._fit(image.convert("RGB"))
        if not self.uses_mask:
            return self.to_tensor(img), None
        if mask is None:
            raise ValueError("scratch mode needs a mask image (white = scratch).")
        mk = mask.convert("RGB").resize(img.size, Image.NEAREST)
        for _ in range(self._mask_dilation):
            mk = mk.filter(ImageFilter.MaxFilter(3))
        m = np.asarray(mk, dtype=np.float32) / 255.0
        holed = np.asarray(img, dtype=np.float32) * (1.0 - m) + 255.0 * m
        img = Image.fromarray(holed.round().clip(0, 255).astype(np.uint8))
        return self.to_tensor(img), self.mask_to_tensor(mk, img.size)

    @staticmethod
    def to_image(x, normalize=True):
        """(1,3,H,W) tensor in [-1,1] -> PIL. normalize=True min-max stretches like
        save_image(..., normalize=True) in Global/test.py; False = plain clamp."""
        y = (x.detach().cpu().float() + 1.0) / 2.0
        if normalize:
            lo, hi = y.min(), y.max()
            y = (y - lo) / (hi - lo).clamp_min(1e-5)
        y = y.clamp(0, 1)[0]
        return Image.fromarray((y.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))

    def restore(self, image, mask=None, keep_size=True, normalize=True):
        """PIL in -> PIL out (full pipeline)."""
        x, m = self.prepare(image, mask)
        out = self.to_image(self.translate(x, m), normalize)
        if keep_size and out.size != image.size:
            out = out.resize(image.size, Image.BICUBIC)
        return out

    # ------------------------------------------------------------------ #
    # Setup: repo, weights, options
    # ------------------------------------------------------------------ #
    @staticmethod
    def _ensure_repo(repo_root):
        if os.path.isfile(os.path.join(repo_root, "Global", "models", "mapping_model.py")):
            return
        os.makedirs(os.path.dirname(repo_root), exist_ok=True)
        with tempfile.TemporaryDirectory() as tmp:
            zpath = os.path.join(tmp, "repo.zip")
            _download(_REPO_ZIP, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(tmp)
            extracted = os.path.join(tmp, "Bringing-Old-Photos-Back-to-Life-master")
            if os.path.exists(repo_root):
                shutil.rmtree(repo_root)
            shutil.move(extracted, repo_root)

    def _ensure_weights(self, global_dir, mode):
        """Download the checkpoint zip (layout: checkpoints/restoration/<folder>/*.pth) and copy
        any missing or empty files for the needed folders."""
        if not self._missing(global_dir, mode):
            return
        target = os.path.join(global_dir, "checkpoints", "restoration")
        wanted = {f for f, _ in _REQUIRED[mode]}
        with tempfile.TemporaryDirectory() as tmp:
            zpath = os.path.join(tmp, "global_checkpoints.zip")
            _download(_CKPT_ZIP, zpath)
            extract_dir = os.path.join(tmp, "extracted")
            with zipfile.ZipFile(zpath) as z:
                z.extractall(extract_dir)
            src_root = os.path.join(extract_dir, "checkpoints", "restoration")
            for folder in wanted:
                os.makedirs(os.path.join(target, folder), exist_ok=True)
                for f in os.listdir(os.path.join(src_root, folder)):
                    src, dst = os.path.join(src_root, folder, f), os.path.join(target, folder, f)
                    if not os.path.isfile(dst) or os.path.getsize(dst) != os.path.getsize(src):
                        shutil.copy2(src, dst)
        self._check_weights(global_dir, mode, raise_error=True)

    @staticmethod
    def _missing(global_dir, mode):
        ckpt = os.path.join(global_dir, "checkpoints", "restoration")
        missing = []
        for folder, label in _REQUIRED[mode]:
            if not glob.glob(os.path.join(ckpt, folder, f"*_net_{label}.pth")):
                missing.append(os.path.join(ckpt, folder, f"latest_net_{label}.pth"))
        return missing

    def _check_weights(self, global_dir, mode, raise_error):
        missing = self._missing(global_dir, mode)
        if missing and raise_error:
            raise FileNotFoundError("Missing checkpoints:\n  " + "\n  ".join(missing) +
                                    f"\nDownload {_CKPT_ZIP} and unzip it inside {global_dir}")
        return missing

    @staticmethod
    def _build_opt(global_dir, mode, gpu):
        if global_dir not in sys.path:
            sys.path.insert(0, global_dir)  # repo imports `options`, `models`, `util` from Global/
        from options.test_options import TestOptions

        flag = "--Quality_restore" if mode == "quality" else "--Scratch_and_Quality_restore"
        backup = sys.argv
        sys.argv = ["old_photo_model", flag, "--gpu_ids", str(gpu)]
        if mode == "scratch_hr":
            sys.argv.append("--HR")
        try:
            opt = TestOptions().parse(save=False)
        finally:
            sys.argv = backup

        # mirrors parameter_set() in Global/test.py
        ckpt = os.path.join(global_dir, "checkpoints", "restoration")
        opt.serial_batches = True
        opt.no_flip = True
        opt.label_nc = 0
        opt.n_downsample_global = 3
        opt.mc = 64
        opt.k_size = 4
        opt.start_r = 1
        opt.mapping_n_block = 6
        opt.map_mc = 512
        opt.no_instance = True
        opt.checkpoints_dir = ckpt

        if mode == "quality":
            opt.name = "mapping_quality"
            opt.load_pretrainA = os.path.join(ckpt, "VAE_A_quality")
            opt.load_pretrainB = os.path.join(ckpt, "VAE_B_quality")
        else:
            opt.NL_res = True
            opt.use_SN = True
            opt.correlation_renormalize = True
            opt.NL_use_mask = True
            opt.NL_fusion_method = "combine"
            opt.non_local = "Setting_42"
            opt.name = "mapping_scratch"
            opt.load_pretrainA = os.path.join(ckpt, "VAE_A_quality")
            opt.load_pretrainB = os.path.join(ckpt, "VAE_B_scratch")
            if mode == "scratch_hr":
                opt.HR = True
                opt.mapping_exp = 1              # Mapping_Model_with_mask_2 (patch attention)
                opt.inference_optimize = True    # enables mapping.inference_forward
                opt.mask_dilation = 3
                opt.name = "mapping_Patch_Attention"
        return opt

    def _verify_weights(self, opt, mode):
        """Re-load each checkpoint with strict=True so a silent partial load can't go unnoticed."""
        ckpt = opt.checkpoints_dir
        vae_epoch = getattr(opt, "use_vae_which_epoch", "latest")
        map_epoch = getattr(opt, "which_epoch", "latest")
        jobs = [
            (self.vae_A, os.path.join(opt.load_pretrainA, f"{vae_epoch}_net_G.pth")),
            (self.vae_B, os.path.join(opt.load_pretrainB, f"{vae_epoch}_net_G.pth")),
            (self.mapping, os.path.join(ckpt, opt.name, f"{map_epoch}_net_mapping_net.pth")),
        ]
        for net, path in jobs:
            try:
                net.load_state_dict(torch.load(path, map_location="cpu"), strict=True)
            except Exception as e:  # noqa: BLE001
                raise RuntimeError(
                    f"Checkpoint {path} does not match the network built for mode '{mode}': {e}\n"
                    "The option flags in _build_opt() probably differ from Global/test.py."
                ) from e

### `degradation.py`

In [ ]:
%%writefile /content/degradation.py
"""
Port of the synthetic degradation pipeline of microsoft/Bringing-Old-Photos-Back-to-Life
(Global/data/online_dataset_for_old_photos.py): blur, Gaussian / speckle / salt-and-pepper noise,
downscaling and JPEG compression, applied in random order.
"""
import random
from io import BytesIO

import cv2
import numpy as np
from PIL import Image


def _pil_to_np(img):
    ar = np.array(img)
    ar = ar.transpose(2, 0, 1) if ar.ndim == 3 else ar[None, ...]
    return ar.astype(np.float32) / 255.0


def _np_to_pil(ar):
    ar = np.clip(ar * 255, 0, 255).astype(np.uint8)
    ar = ar[0] if ar.shape[0] == 1 else ar.transpose(1, 2, 0)
    return Image.fromarray(ar)


def synthesize_salt_pepper(image, amount, salt_vs_pepper):
    img = _pil_to_np(image)
    out = img.copy()
    flipped = np.random.choice([True, False], size=img.shape, p=[amount, 1 - amount])
    salted = np.random.choice([True, False], size=img.shape, p=[salt_vs_pepper, 1 - salt_vs_pepper])
    peppered = ~salted
    out[flipped & salted] = 1
    out[flipped & peppered] = 0.0
    return _np_to_pil(np.clip(out, 0, 1).astype(np.float32))


def synthesize_gaussian(image, std_l, std_r):
    img = _pil_to_np(image)
    std = random.uniform(std_l / 255.0, std_r / 255.0)
    noisy = img + np.random.normal(0, std, size=img.shape)
    return _np_to_pil(np.clip(noisy, 0, 1).astype(np.float32))


def synthesize_speckle(image, std_l, std_r):
    img = _pil_to_np(image)
    std = random.uniform(std_l / 255.0, std_r / 255.0)
    noisy = img + np.random.normal(0, std, size=img.shape) * img
    return _np_to_pil(np.clip(noisy, 0, 1).astype(np.float32))


def synthesize_low_resolution(img):
    w, h = img.size
    new_w = random.randint(int(w / 2), w)
    new_h = random.randint(int(h / 2), h)
    img = img.resize((new_w, new_h), Image.BICUBIC)
    method = Image.NEAREST if random.uniform(0, 1) < 0.5 else Image.BILINEAR
    return img.resize((w, h), method)


def convert_to_jpeg(img, quality):
    with BytesIO() as f:
        img.save(f, format="JPEG", quality=quality)
        f.seek(0)
        return Image.open(f).convert("RGB")


def blur_image_v2(img):
    x = np.array(img)
    kernel_size = random.choice([(3, 3), (5, 5), (7, 7)])
    std = random.uniform(1.0, 5.0)
    blur = cv2.GaussianBlur(x, kernel_size, std)
    return Image.fromarray(blur.astype(np.uint8))


def online_add_degradation_v2(img):
    """The repo's exact composite degradation used for training domain A / the mapping net."""
    task_id = np.random.permutation(4)
    for x in task_id:
        if x == 0 and random.uniform(0, 1) < 0.7:
            img = blur_image_v2(img)
        if x == 1 and random.uniform(0, 1) < 0.7:
            flag = random.choice([1, 2, 3])
            if flag == 1:
                img = synthesize_gaussian(img, 5, 50)
            elif flag == 2:
                img = synthesize_speckle(img, 5, 50)
            elif flag == 3:
                img = synthesize_salt_pepper(img, random.uniform(0, 0.01), random.uniform(0.3, 0.8))
        if x == 2 and random.uniform(0, 1) < 0.7:
            img = synthesize_low_resolution(img)
        if x == 3 and random.uniform(0, 1) < 0.7:
            img = convert_to_jpeg(img, random.randint(40, 100))
    return img


def irregular_hole_synthesize(img, mask):
    """Paint the masked (white) region white, exactly as the repo does before encoding domain A in scratch mode."""
    img_np = np.array(img).astype("uint8")
    mask_np = np.array(mask).astype("uint8") / 255
    img_new = img_np * (1 - mask_np) + mask_np * 255
    hole_img = Image.fromarray(img_new.astype("uint8")).convert("RGB")
    return hole_img, mask.convert("L")

### `online_dataset.py`

In [ ]:
%%writefile /content/online_dataset.py
"""
Training pairs generated on the fly: every __getitem__ draws a fresh clean image, a fresh random crop and a
fresh random degradation, as in the original BOPBTL training, so the model never sees the same degraded
instance twice.

Validation uses a fixed, in-memory set built once with a fixed seed from held-out source images (never used
for training), so validation numbers are comparable across runs.
"""
import glob
import math
import os
import random

import torch
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from degradation import online_add_degradation_v2

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".ppm")
FINE_SIZE = 256

_to_tensor_norm = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # -> [-1,1]
])


def list_clean_images(clean_dir, min_side=256):
    files = sorted(f for f in glob.glob(os.path.join(clean_dir, "*")) if f.lower().endswith(IMG_EXT))
    kept, corrupt = [], 0
    for f in files:
        try:
            with Image.open(f) as im:
                im.verify()          # catches truncated/corrupt files
            with Image.open(f) as im:  # verify() invalidates the handle, reopen to read size
                w, h = im.size
        except Exception:
            corrupt += 1
            continue
        if w >= min_side and h >= min_side:
            kept.append(f)
    print(f"{len(kept)}/{len(files)} images pass the >= {min_side}px filter "
          f"({corrupt} unreadable/corrupt files skipped)")
    return kept


def _crop_params(size, fine_size=FINE_SIZE):
    w, h = size
    x = random.randint(0, max(0, w - fine_size))
    y = random.randint(0, max(0, h - fine_size))
    flip = random.random() > 0.5
    return (x, y), flip


def _apply_crop_flip(img, pos, flip, fine_size=FINE_SIZE):
    ow, oh = img.size
    x1, y1 = pos
    if ow > fine_size or oh > fine_size:
        img = img.crop((x1, y1, x1 + fine_size, y1 + fine_size))
    if flip:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    return img


def make_online_pair(clean_path, grayscale_prob=0.1):
    """Fresh (A, B) tensor pair every call: new crop position, new flip, new
    degradation realization -- even for the same source image.

    grayscale_prob: fraction of pairs where BOTH A and B are desaturated
    (matches the repo's own training, which also keeps B grayscale when A is).
    Real target old photos are overwhelmingly B&W, so the default 0.1 (matching
    the original repo) under-represents that case for a mapping net trained
    from scratch without the repo's additional real-old-photo VAE_A domain
    adaptation -- raise this for training runs targeting B&W restoration."""
    B = Image.open(clean_path).convert("RGB")
    A = online_add_degradation_v2(B)
    if A.size != B.size:
        A = A.resize(B.size, Image.BILINEAR)

    if random.uniform(0, 1) < grayscale_prob:
        A = A.convert("L").convert("RGB")
        B = B.convert("L").convert("RGB")

    pos, flip = _crop_params(B.size)
    A = _apply_crop_flip(A, pos, flip)
    B = _apply_crop_flip(B, pos, flip)

    if A.size != (FINE_SIZE, FINE_SIZE):
        A = A.resize((FINE_SIZE, FINE_SIZE), Image.BICUBIC)
        B = B.resize((FINE_SIZE, FINE_SIZE), Image.BICUBIC)

    return _to_tensor_norm(A), _to_tensor_norm(B)


class OnlineQualityDataset(Dataset):
    """len() is nominal (controls how many samples count as "one epoch" for
    DataLoader bookkeeping / shuffling), but __getitem__ always draws a fresh
    random source image + fresh degradation, regardless of idx. This is the
    key difference from a dataset that reads a fixed PNG from disk per idx."""

    def __init__(self, clean_dir, length=20000, min_side=256, grayscale_prob=0.1):
        self.files = list_clean_images(clean_dir, min_side)
        assert self.files, f"no usable images (>= {min_side}px both sides) in {clean_dir}"
        self.length = length
        self.grayscale_prob = grayscale_prob

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        for _ in range(5):  # a handful of retries in case of a corrupt/transient-unreadable file
            f = random.choice(self.files)
            try:
                return make_online_pair(f, self.grayscale_prob)
            except Exception as e:
                print(f"[OnlineQualityDataset] skipping unreadable {f}: {e}")
        raise RuntimeError("too many consecutive unreadable files -- check your image folder")


def make_online_train_loader(clean_dir, batch_size=16, epoch_length=20000, num_workers=4, grayscale_prob=0.1):
    """Online training loader: every sample is freshly degraded. Use a
    multi-worker DataLoader since degradation (PIL/cv2 ops) is CPU-bound and
    would otherwise bottleneck the GPU."""
    ds = OnlineQualityDataset(clean_dir, length=epoch_length, grayscale_prob=grayscale_prob)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0,
    )
    print(f"online train loader: {len(ds.files)} source images, "
          f"{epoch_length} samples/epoch, {len(loader)} batches/epoch")
    return loader


# --------------------------------------------------------------------------- #
# Train/val split at the SOURCE-IMAGE level, both drawn from the same folder.
# --------------------------------------------------------------------------- #
class FixedPairDataset(Dataset):
    """A small, pre-generated, in-memory set of (A, B) tensor pairs. Built
    once with a fixed seed so it's identical across runs/evals, but drawn
    from source images that were NEVER used for training."""

    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


def make_online_dataloaders(clean_dir, batch_size=16, epoch_length=20000,
                             num_workers=2, val_fraction=0.05, val_samples=400, seed=0,
                             grayscale_prob=0.1):
    """Splits source images into train/val pools (no overlap), builds the
    online training loader from the train pool, and a FIXED val set (built
    once, with a fixed seed, same degradation pipeline) from the held-out
    val pool -- so val is comparable across runs AND actually measures
    generalization on the same visual domain the model is training on."""
    all_files = list_clean_images(clean_dir)
    rng = random.Random(seed)
    shuffled = all_files[:]
    rng.shuffle(shuffled)
    n_val_files = max(1, int(len(shuffled) * val_fraction))
    val_files, train_files = shuffled[:n_val_files], shuffled[n_val_files:]
    print(f"split: {len(train_files)} train source images, {len(val_files)} val source images (no overlap)")

    train_ds = OnlineQualityDataset.__new__(OnlineQualityDataset)  # bypass list_clean_images re-scan
    train_ds.files = train_files
    train_ds.length = epoch_length
    train_ds.grayscale_prob = grayscale_prob
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0,
    )

    # build the fixed val set once, with a local seeded RNG so it's reproducible
    state = random.getstate()
    random.seed(seed + 1)
    pairs = []
    i = 0
    while len(pairs) < val_samples:
        f = val_files[i % len(val_files)]
        try:
            pairs.append(make_online_pair(f, grayscale_prob))
        except Exception as e:
            print(f"[val set] skipping unreadable {f}: {e}")
        i += 1
    random.setstate(state)

    val_loader = DataLoader(FixedPairDataset(pairs), batch_size=batch_size, shuffle=False, num_workers=0)
    print(f"online train loader: {len(train_files)} source images, {epoch_length} samples/epoch, "
          f"{len(train_loader)} batches/epoch | fixed val: {len(pairs)} samples from {len(val_files)} held-out images")
    return train_loader, val_loader

### `rf_model.py`

In [ ]:
%%writefile /content/rf_model.py
"""
Rectified flow model mapping z_A -> z_B in the frozen VAE latent space.

VelocityNet: v_theta(z_t, t | z_A), fully convolutional (works at any latent
resolution). FiLM-conditioned residual blocks, dilated convs for receptive
field, zero-initialized output so the flow starts as the identity.

RFMapping: wraps VelocityNet with the scalar latent normalization and the
Euler ODE integrator used both at train time (for validation samples) and at
inference time (obj.mapping replacement).

Bridge matching (sigma > 0): the straight path gets Brownian-bridge noise,
    z_t = (1 - t) z_A + t z_B + sigma * sqrt(t (1 - t)) * eps,
zero at both ends. The network output v defines the predicted endpoint
z_B_hat = z_t + (1 - t) v, trained against the true z_B, and sampling uses the
exact bridge posterior step
    z <- z + dt / (1 - t) * (z_B_hat - z) + sigma * sqrt(dt (1 - t') / (1 - t)) * eps.
A deterministic flow (sigma = 0) regresses to the AVERAGE of all clean images
consistent with z_A (soft texture, washed-out colour); with sigma > 0 the
sampler draws ONE plausible z_B instead. At sigma = 0 the sampler is exactly
the old Euler RF sampler, so RF checkpoints warm-start a bridge model directly.

direct_loss() (t = 0 ablation): only the point the 1-step sampler uses, z_t = z_A,
t = 0 -- plain direct regression z_A -> z_B with the same network, i.e. the
Microsoft-style mapping objective. No bridge noise, no t > 0.

traj_loss() trains the velocity at points the sampler actually reaches
(the model's own Euler trajectory), not only on the ideal straight line, so
the model learns to correct its own drift instead of accumulating it.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class TimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):  # t: (B,) in [0,1]
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = (t[:, None] * 1000.0) * freqs[None]
        return self.mlp(torch.cat([args.sin(), args.cos()], dim=1))


class FiLMResBlock(nn.Module):
    def __init__(self, channels, tdim, dilation=1, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.film = nn.Linear(tdim, 2 * channels)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        scale, shift = self.film(t_emb)[:, :, None, None].chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.dropout(F.silu(h))
        return x + self.conv2(h)


class VelocityNet(nn.Module):
    """v(z_t, t | z_A). Input is concat(z_t, z_A) along channels."""

    def __init__(self, latent_ch, width=256, n_blocks=9, tdim=256, dropout=0.1):
        super().__init__()
        self.time_emb = TimeEmb(tdim)
        self.in_conv = nn.Conv2d(2 * latent_ch, width, 3, padding=1)
        dilations = (1, 2, 4)
        self.blocks = nn.ModuleList([
            FiLMResBlock(width, tdim, dilation=dilations[i % len(dilations)], dropout=dropout)
            for i in range(n_blocks)
        ])
        self.out_conv = nn.Conv2d(width, latent_ch, 3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, z_t, t, z_A):
        t_emb = self.time_emb(t)
        h = self.in_conv(torch.cat([z_t, z_A], dim=1))
        for block in self.blocks:
            h = block(h, t_emb)
        return self.out_conv(F.silu(h))


class RFMapping(nn.Module):
    """Rectified-flow replacement for the repo's mapping net.

    forward(z_A) integrates dz/dt = v(z, t | z_A) from t=0 (z_A) to t=1 (z_B)
    with `steps` Euler steps, matching the OldPhotoModel.mapping call signature
    closely enough to drop in via rf_mapping.install() / obj.mapping = rf.
    """

    def __init__(self, latent_ch, width=256, n_blocks=9, steps=8, sigma=0.0):
        super().__init__()
        self.latent_ch = latent_ch
        self.width = width
        self.n_blocks = n_blocks
        self.v = VelocityNet(latent_ch, width, n_blocks)
        self.steps = steps
        self.sigma = float(sigma)     # bridge noise (0 = deterministic RF); stored in checkpoint["arch"]
        self.sample_sigma = None      # inference override: None = use self.sigma, 0 = deterministic
        self.register_buffer("scale", torch.ones(()))  # latent std, set from training data

    def forward(self, z_A, mask=None, steps=None):
        return self.sample(z_A, steps)

    @staticmethod
    def _noise(z, generator=None):
        if generator is None:
            return torch.randn_like(z)
        return torch.randn(z.shape, generator=generator, device=z.device, dtype=z.dtype)

    @torch.no_grad()
    def sample(self, z_A, steps=None, sigma=None, generator=None):
        """n bridge-posterior steps from z_A. sigma: noise level (default:
        sample_sigma if set, else the trained sigma). sigma=0 is the old Euler
        RF sampler exactly. generator: optional torch.Generator for repeatable noise."""
        n = steps or self.steps
        if sigma is None:
            sigma = self.sigma if self.sample_sigma is None else self.sample_sigma
        a = z_A / self.scale
        z = a.clone()
        for i in range(n):
            t, t2 = i / n, (i + 1) / n
            z = z + self.v(z, torch.full((z.shape[0],), t, device=z.device), a) / n   # = z + dt/(1-t) (z_B_hat - z)
            if sigma > 0 and i < n - 1:
                z = z + sigma * math.sqrt((t2 - t) * (1 - t2) / (1 - t)) * self._noise(z, generator)
        return z * self.scale

    def rf_loss(self, z_A, z_B, time_sampler="logit_normal", return_endpoint=False):
        """Rectified-flow training loss: MSE between v_theta and the straight-line
        velocity (z_B - z_A), evaluated at a random point on the line.

        return_endpoint=True also returns z_B_hat = z_t + (1 - t) * v, the
        model's one-jump guess of z_B from that point (unscaled, differentiable),
        for image-space losses on decode_B(z_B_hat)."""
        a = z_A / self.scale
        b = z_B / self.scale
        bsz = a.shape[0]
        if time_sampler == "logit_normal":
            t = torch.sigmoid(torch.randn(bsz, device=a.device))
        else:
            t = torch.rand(bsz, device=a.device)
        tt = t[:, None, None, None]
        z_t = (1 - tt) * a + tt * b
        if self.sigma > 0:
            # bridge: noise on the path; target = drift towards z_B from the noisy point.
            # (b - z_t) / (1 - t) has noise / sqrt(1 - t), so the error is weighted by
            # (1 - t)^2 (= endpoint-space error), normalized to mean 1, as in traj_loss.
            t = t.clamp(max=0.999)
            tt = t[:, None, None, None]
            z_t = z_t + self.sigma * torch.sqrt(tt * (1 - tt)) * torch.randn_like(z_t)
            target = (b - z_t) / (1 - tt)
            pred = self.v(z_t, t, a)
            w = (1 - t) ** 2
            w = w / w.mean()
            loss = (w * ((pred - target) ** 2).flatten(1).mean(1)).mean()
        else:
            target = b - a
            pred = self.v(z_t, t, a)
            loss = F.mse_loss(pred, target)
        if not return_endpoint:
            return loss
        z_B_hat = (z_t + (1 - tt) * pred) * self.scale
        return loss, z_B_hat

    def direct_loss(self, z_A, z_B, return_endpoint=False):
        """t = 0 only: v(z_A, 0 | z_A) regresses z_B - z_A, so z_B_hat = z_A + v is
        exactly what sample(steps=1) returns. Same MSE (scaled latent) as rf_loss,
        which at t = 0 has the same target and weight."""
        a = z_A / self.scale
        b = z_B / self.scale
        t = torch.zeros(a.shape[0], device=a.device)
        pred = self.v(a, t, a)
        loss = F.mse_loss(pred, b - a)
        if not return_endpoint:
            return loss
        return loss, (a + pred) * self.scale

    @torch.no_grad()
    def rollout(self, a, k, n, sigma=0.0):
        """Per-sample partial sampling in scaled space: sample i takes the first
        k[i] of n[i] steps from a[i] (with bridge noise if sigma > 0, exactly as
        sample() does). a: (B,C,H,W) = z_A / scale; k, n: (B,) float tensors
        with k <= n - 1. Returns the reached points z."""
        z = a.clone()
        for i in range(int(k.max().item())):
            active = (i < k)[:, None, None, None]
            # finished samples (i >= k) are still in the batch: clamp their time below 1 so nothing
            # divides by zero, and select with where() -- 0 * nan would still be nan
            t = torch.minimum(torch.full_like(n, float(i)), n - 1) / n
            t2 = t + 1 / n
            step = self.v(z, t, a) / n[:, None, None, None]
            if sigma > 0:
                std = sigma * torch.sqrt((t2 - t) * (1 - t2) / (1 - t))
                step = step + std[:, None, None, None] * torch.randn_like(z)
            z = torch.where(active, z + step, z)
        return z

    def traj_loss(self, z_A, z_B, roller, step_choices=(4, 8, 16), return_endpoint=False, n_mode="sample"):
        """Like rf_loss, but at a point on the model's OWN trajectory: `roller`
        (normally the EMA copy, i.e. the weights used at inference) runs k of n
        Euler steps from z_A, with n drawn from step_choices and k uniform in
        [1, n-1] per sample. The target is the velocity that reaches the true
        z_B from there in the remaining time: (z_B - z) / (1 - t).

        The per-sample error is weighted by (1 - t)^2, i.e. measured where the
        jump lands ((z + (1-t) v) vs z_B) rather than in velocity, and the
        weights are normalized to mean 1. Without it, 1 / (1 - t) up to n
        amplifies drift late in the path (x256 in MSE at n=16) and those few
        samples dominate the loss.

        n_mode="batch" draws ONE sampler length n for the whole batch instead of
        one per sample. The rollout runs until the largest k in the batch, so
        with long samplers (e.g. 64) in the choices, per-sample n makes almost
        every batch pay ~n forward passes; per-batch n costs ~2.5x less on
        average and covers the same (n, k) distribution over training."""
        a = z_A / self.scale
        b = z_B / self.scale
        bsz = a.shape[0]
        choices = torch.tensor(step_choices, dtype=torch.float32, device=a.device)
        if n_mode == "batch":
            n = choices[torch.randint(len(step_choices), (1,), device=a.device)].expand(bsz)
        else:
            n = choices[torch.randint(len(step_choices), (bsz,), device=a.device)]
        k = torch.floor(torch.rand(bsz, device=a.device) * (n - 1)) + 1
        z = roller.rollout(a, k, n, self.sigma)
        t = k / n
        tt = t[:, None, None, None]
        target = (b - z) / (1 - tt)
        pred = self.v(z, t, a)
        w = (1 - t) ** 2
        w = w / w.mean()
        loss = (w * ((pred - target) ** 2).flatten(1).mean(1)).mean()
        if not return_endpoint:
            return loss
        z_B_hat = (z + (1 - tt) * pred) * self.scale
        return loss, z_B_hat

    def clone_for_ema(self):
        ema = RFMapping(self.latent_ch, self.width, self.n_blocks, self.steps, self.sigma)
        ema.load_state_dict(self.state_dict())
        for p in ema.parameters():
            p.requires_grad_(False)
        return ema

    @torch.no_grad()
    def ema_update(self, source, decay=0.999):
        for p_ema, p in zip(self.v.parameters(), source.v.parameters()):
            p_ema.mul_(decay).add_(p, alpha=1 - decay)
        self.scale.copy_(source.scale)  # scalar: just mirror, no EMA needed

### `direct_model.py`

In [ ]:
%%writefile /content/direct_model.py
"""
DirectMapping: the final z_A -> z_B mapping, a plain residual conv network in the
frozen VAE latent space (same mechanism as Microsoft's mapping network, ~3x smaller).

It is rf_model.VelocityNet with the time input removed. With t fixed to 0:
  - the time embedding is a constant vector, so each FiLM layer is a constant
    per-channel scale/shift right after norm2, which norm2's own affine absorbs;
  - the input concat(z_t, z_A) is concat(z_A, z_A), so the first conv's two weight
    halves merge into one conv on z_A.
So the function class is exactly that of the t = 0 model, and fold_rf() turns a
trained t = 0 RFMapping into a DirectMapping with identical outputs.

    z_B = scale * (a + net(a)),   a = z_A / scale

Residual, with a zero-initialized output conv, so training starts at the identity.
forward(z_A, mask=None) matches OldPhotoModel.mapping's call signature: obj.mapping = net.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    """FiLMResBlock without FiLM: GroupNorm -> SiLU -> dilated conv -> GroupNorm -> SiLU -> dropout -> conv, + skip."""

    def __init__(self, channels, dilation=1, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.dropout(F.silu(self.norm2(h)))
        return x + self.conv2(h)


class DirectMapping(nn.Module):
    def __init__(self, latent_ch, width=256, n_blocks=9, dropout=0.1):
        super().__init__()
        self.latent_ch, self.width, self.n_blocks = latent_ch, width, n_blocks
        self.in_conv = nn.Conv2d(latent_ch, width, 3, padding=1)
        dilations = (1, 2, 4)
        self.blocks = nn.ModuleList([ResBlock(width, dilations[i % len(dilations)], dropout)
                                     for i in range(n_blocks)])
        self.out_conv = nn.Conv2d(width, latent_ch, 3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)
        self.register_buffer("scale", torch.ones(()))  # latent std, set from training data

    def forward(self, z_A, mask=None):
        a = z_A / self.scale
        h = self.in_conv(a)
        for block in self.blocks:
            h = block(h)
        return (a + self.out_conv(F.silu(h))) * self.scale

    def arch(self):
        return {"type": "direct", "latent_ch": self.latent_ch, "width": self.width, "n_blocks": self.n_blocks}


@torch.no_grad()
def fold_rf(rf):
    """t = 0 RFMapping -> DirectMapping with identical outputs (both in eval mode)."""
    v, c = rf.v, rf.latent_ch
    d = DirectMapping(c, rf.width, rf.n_blocks).to(rf.scale.device)
    d.scale.copy_(rf.scale)
    w = v.in_conv.weight                                  # (width, 2c, 3, 3): halves for z_t and z_A
    d.in_conv.weight.copy_(w[:, :c] + w[:, c:])
    d.in_conv.bias.copy_(v.in_conv.bias)
    t_emb = v.time_emb(torch.zeros(1, device=w.device))   # constant at t = 0
    for bd, bv in zip(d.blocks, v.blocks):
        scale, shift = bv.film(t_emb)[0].chunk(2)
        for name in ("norm1", "conv1", "conv2"):
            getattr(bd, name).load_state_dict(getattr(bv, name).state_dict())
        # norm2(h) * (1 + scale) + shift = gn(h) * w (1 + scale) + (b (1 + scale) + shift)
        bd.norm2.weight.copy_(bv.norm2.weight * (1 + scale))
        bd.norm2.bias.copy_(bv.norm2.bias * (1 + scale) + shift)
    d.out_conv.load_state_dict(v.out_conv.state_dict())
    return d.eval()


def load_direct(path, device, use_ema=True):
    """DirectMapping from a train_direct checkpoint (or the exported weights file). Returns (net, step)."""
    ck = torch.load(path, map_location=device)
    arch = {k: v for k, v in ck["arch"].items() if k != "type"}
    net = DirectMapping(**arch).to(device)
    net.load_state_dict(ck["ema"] if use_ema else ck["net"])
    net.eval()
    print(f"loaded {path} | step {ck['step']} | {sum(p.numel() for p in net.parameters()) / 1e6:.2f}M params")
    return net, ck["step"]

### `benchmark.py`

In [ ]:
%%writefile /content/benchmark.py
"""
Large, independent, frozen benchmark for the old-photo mapping.

build_benchmark(): takes clean photos the models never saw (e.g. COCO val2017 or
DIV2K valid), makes degraded/clean 256x256 pairs with the SAME synthetic
degradation as training (make_online_pair: online_add_degradation_v2, crop,
50% gray), each pair seeded by its index, and saves them as PNGs + meta.csv.
Built once (resumable), so every model is scored on exactly the same pairs.

evaluate_benchmark(): streams the pairs from disk and reports, per method,
PSNR, SSIM, LPIPS, FID, KID (unbiased at any n; mean +- std over subsets),
CIEDE2000 colour error, a gray / colour split, and mapping cost. Reference rows:
degraded input, Microsoft mapping, and the VAE B ceiling (clean -> VAE B -> clean,
the best any mapping can do).

FID / KID are computed between each method's outputs and the PAIRED clean targets
(same scenes). FID values are therefore only comparable between methods on the
same benchmark, never with other papers or other n; KID is unbiased, so its
differences can be judged against its std. (A clean-vs-clean comparison of two
disjoint halves is NOT a valid floor here: it measures scene differences too,
which is why it can exceed the ceiling's FID.)
"""
import csv
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from skimage.metrics import structural_similarity

from online_dataset import list_clean_images, make_online_pair


# --------------------------------------------------------------------------- #
# Building the benchmark
# --------------------------------------------------------------------------- #
def _to_uint8(t):
    """(3,H,W) in [-1,1] -> (H,W,3) uint8."""
    return ((t.clamp(-1, 1) + 1) * 127.5).round().byte().permute(1, 2, 0).numpy()


def build_benchmark(src_dir, out_dir, n=5000, crops_per_image=1, seed=0, grayscale_prob=0.5):
    """Saves n degraded/clean pairs: out_dir/A/000000.png, out_dir/B/000000.png, meta.csv.
    Resumable: pairs already listed in meta.csv are skipped. Pair i uses source image
    i // crops_per_image (of a seeded shuffle) and its own fixed random seed."""
    os.makedirs(f"{out_dir}/A", exist_ok=True)
    os.makedirs(f"{out_dir}/B", exist_ok=True)
    meta_path = f"{out_dir}/meta.csv"
    files = list_clean_images(src_dir)
    random.Random(seed).shuffle(files)
    total = min(n, len(files) * crops_per_image)
    done = set()
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            done = {int(r["idx"]) for r in csv.DictReader(f)}
    print(f"benchmark: {total} pairs from {len(files)} source images ({len(done)} already built)")
    new = not os.path.exists(meta_path)
    with open(meta_path, "a", newline="") as mf:
        w = csv.writer(mf)
        if new:
            w.writerow(["idx", "source", "gray"])
        t0, made = time.time(), 0
        for i in range(total):
            if i in done:
                continue
            src = files[i // crops_per_image]
            s = seed * 1_000_003 + i
            random.seed(s)
            np.random.seed(s % 2 ** 32)
            try:
                a, b = make_online_pair(src, grayscale_prob)
            except Exception as e:
                print(f"  skipping {src}: {e}")
                continue
            gray = bool((b - b.mean(0, keepdim=True)).abs().max() < 1e-3)
            Image.fromarray(_to_uint8(a)).save(f"{out_dir}/A/{i:06d}.png")
            Image.fromarray(_to_uint8(b)).save(f"{out_dir}/B/{i:06d}.png")
            w.writerow([i, os.path.basename(src), int(gray)])
            made += 1
            if made % 250 == 0:
                mf.flush()
                print(f"  {len(done) + made}/{total} pairs ({made / (time.time() - t0):.1f}/s)")
    print(f"benchmark ready: {out_dir}")


class BenchPairs(torch.utils.data.Dataset):
    """(A, B, gray) from a built benchmark; A, B in [-1,1]."""

    def __init__(self, bench_dir, max_images=None):
        self.dir = bench_dir
        with open(f"{bench_dir}/meta.csv") as f:
            rows = sorted(csv.DictReader(f), key=lambda r: int(r["idx"]))
        self.rows = rows[:max_images] if max_images else rows

    def __len__(self):
        return len(self.rows)

    def _load(self, sub, idx):
        a = np.asarray(Image.open(f"{self.dir}/{sub}/{idx:06d}.png").convert("RGB"), dtype=np.float32)
        return torch.from_numpy(a / 127.5 - 1.0).permute(2, 0, 1)

    def __getitem__(self, k):
        idx = int(self.rows[k]["idx"])
        return self._load("A", idx), self._load("B", idx), int(self.rows[k]["gray"])


# --------------------------------------------------------------------------- #
# Colour science (same code as color_analysis.py, checked against scikit-image)
# --------------------------------------------------------------------------- #
_M_RGB2XYZ = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                           [0.2126729, 0.7151522, 0.0721750],
                           [0.0193339, 0.1191920, 0.9503041]])
_WHITE_D65 = torch.tensor([0.95047, 1.0, 1.08883])


def srgb_to_lab(rgb):
    """(N,3,H,W) sRGB in [0,1] -> (N,3,H,W) CIELAB (D65, 2 deg), like skimage.color.rgb2lab."""
    rgb = rgb.clamp(0, 1)
    lin = torch.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    xyz = torch.einsum("ij,njhw->nihw", _M_RGB2XYZ.to(rgb), lin) / _WHITE_D65.to(rgb)[None, :, None, None]
    eps = (6 / 29) ** 3
    f = torch.where(xyz > eps, xyz.clamp_min(1e-12) ** (1 / 3), xyz / (3 * (6 / 29) ** 2) + 4 / 29)
    L = 116 * f[:, 1] - 16
    a = 500 * (f[:, 0] - f[:, 1])
    b = 200 * (f[:, 1] - f[:, 2])
    return torch.stack([L, a, b], 1)


def ciede2000(lab1, lab2):
    """Per-pixel CIEDE2000 between (N,3,H,W) Lab tensors -> (N,H,W). (Sharma et al. 2005)"""
    L1, a1, b1 = lab1[:, 0], lab1[:, 1], lab1[:, 2]
    L2, a2, b2 = lab2[:, 0], lab2[:, 1], lab2[:, 2]
    C1 = torch.sqrt(a1 ** 2 + b1 ** 2)
    C2 = torch.sqrt(a2 ** 2 + b2 ** 2)
    Cb7 = ((C1 + C2) / 2) ** 7
    G = 0.5 * (1 - torch.sqrt(Cb7 / (Cb7 + 25.0 ** 7)))
    a1p, a2p = (1 + G) * a1, (1 + G) * a2
    C1p = torch.sqrt(a1p ** 2 + b1 ** 2)
    C2p = torch.sqrt(a2p ** 2 + b2 ** 2)
    h1p = torch.rad2deg(torch.atan2(b1, a1p)) % 360
    h2p = torch.rad2deg(torch.atan2(b2, a2p)) % 360
    zero = (C1p * C2p) == 0

    dLp = L2 - L1
    dCp = C2p - C1p
    dh = h2p - h1p
    dh = torch.where(dh > 180, dh - 360, torch.where(dh < -180, dh + 360, dh))
    dh = torch.where(zero, torch.zeros_like(dh), dh)
    dHp = 2 * torch.sqrt(C1p * C2p) * torch.sin(torch.deg2rad(dh / 2))

    Lbp = (L1 + L2) / 2
    Cbp = (C1p + C2p) / 2
    hsum = h1p + h2p
    hbp = torch.where((h1p - h2p).abs() <= 180, hsum / 2,
                      torch.where(hsum < 360, (hsum + 360) / 2, (hsum - 360) / 2))
    hbp = torch.where(zero, hsum, hbp)
    T = (1 - 0.17 * torch.cos(torch.deg2rad(hbp - 30)) + 0.24 * torch.cos(torch.deg2rad(2 * hbp))
         + 0.32 * torch.cos(torch.deg2rad(3 * hbp + 6)) - 0.20 * torch.cos(torch.deg2rad(4 * hbp - 63)))
    dtheta = 30 * torch.exp(-(((hbp - 275) / 25) ** 2))
    Cbp7 = Cbp ** 7
    RC = 2 * torch.sqrt(Cbp7 / (Cbp7 + 25.0 ** 7))
    SL = 1 + 0.015 * (Lbp - 50) ** 2 / torch.sqrt(20 + (Lbp - 50) ** 2)
    SC = 1 + 0.045 * Cbp
    SH = 1 + 0.015 * Cbp * T
    RT = -torch.sin(torch.deg2rad(2 * dtheta)) * RC
    return torch.sqrt(((dLp / SL) ** 2 + (dCp / SC) ** 2 + (dHp / SH) ** 2
                       + RT * (dCp / SC) * (dHp / SH)).clamp_min(0))


# --------------------------------------------------------------------------- #
# Distribution metrics
# --------------------------------------------------------------------------- #
def fid_from_feats(f1, f2):
    from pytorch_fid.fid_score import calculate_frechet_distance
    f1, f2 = f1.astype(np.float64), f2.astype(np.float64)
    return float(calculate_frechet_distance(f1.mean(0), np.cov(f1, rowvar=False),
                                            f2.mean(0), np.cov(f2, rowvar=False)))


def kid_from_feats(f1, f2, n_subsets=100, subset_size=1000, seed=0):
    """Kernel Inception Distance (Binkowski et al. 2018): unbiased MMD^2 with the
    cubic polynomial kernel k(x,y) = (x.y/d + 1)^3, averaged over random subsets.
    Returns (mean, std). Unlike FID it is unbiased, so it is comparable across n."""
    rng = np.random.default_rng(seed)
    d = f1.shape[1]
    m = min(subset_size, len(f1), len(f2))
    vals = []
    for _ in range(n_subsets):
        x = f1[rng.choice(len(f1), m, replace=False)].astype(np.float64)
        y = f2[rng.choice(len(f2), m, replace=False)].astype(np.float64)
        kxx = (x @ x.T / d + 1) ** 3
        kyy = (y @ y.T / d + 1) ** 3
        kxy = (x @ y.T / d + 1) ** 3
        mmd = ((kxx.sum() - np.trace(kxx)) + (kyy.sum() - np.trace(kyy))) / (m * (m - 1)) - 2 * kxy.mean()
        vals.append(mmd)
    return float(np.mean(vals)), float(np.std(vals))


# --------------------------------------------------------------------------- #
# Evaluation
# --------------------------------------------------------------------------- #
class _Metrics:
    def __init__(self, device):
        import lpips
        from pytorch_fid.inception import InceptionV3
        self.lpips = lpips.LPIPS(net="alex", verbose=False).to(device).eval()
        self.inception = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(device).eval()

    @torch.no_grad()
    def per_image(self, y, t):
        """y, t: (N,3,H,W) in [0,1] -> dict of per-image metrics (CPU)."""
        mse = ((y - t) ** 2).flatten(1).mean(1)
        p = y.permute(0, 2, 3, 1).cpu().numpy()
        q = t.permute(0, 2, 3, 1).cpu().numpy()
        ssim = [structural_similarity(q[k], p[k], channel_axis=2, data_range=1.0, gaussian_weights=True,
                                      sigma=1.5, use_sample_covariance=False) for k in range(len(p))]
        return {"psnr": (10 * torch.log10(1.0 / mse.clamp_min(1e-10))).cpu(),
                "ssim": torch.tensor(ssim, dtype=torch.float32),
                "lpips": self.lpips(y * 2 - 1, t * 2 - 1).flatten().cpu(),
                "de00": ciede2000(srgb_to_lab(t), srgb_to_lab(y)).flatten(1).mean(1).cpu()}

    @torch.no_grad()
    def feats(self, y):
        return self.inception(y)[0].flatten(1).cpu().numpy()


@torch.no_grad()
def evaluate_benchmark(obj, rfs, bench_dir, rf_steps=(1,), batch_size=16, max_images=None,
                       include_input=True, include_ceiling=True, out_csv=None, num_workers=2,
                       include_microsoft=True):
    """obj: OldPhotoModel (quality); rfs: {"name": RFMapping}. Bridge noise is seeded per method."""
    device = obj.device
    metrics = _Metrics(device)
    ds = BenchPairs(bench_dir, max_images)
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    print(f"evaluating on {len(ds)} pairs from {bench_dir}")

    methods = []
    if include_input:
        methods.append(("degraded input", "input", None, None))
    if include_microsoft:
        methods.append(("Microsoft mapping", "net", obj._core.mapping_net, None))
    for name, rf in rfs.items():
        for s in rf_steps:            # s=None: a direct mapping (no sampler), labelled by its name only
            methods.append((name if s is None else f"{name} ({s}-step)", "net", rf, s))
    if include_ceiling:
        methods.append(("VAE B ceiling", "ceiling", None, None))

    to01 = lambda x: ((x.float() + 1) / 2).clamp(0, 1)
    real_feats, gray_all, rows = None, None, []
    for name, kind, net, steps in methods:
        per = {k: [] for k in ("psnr", "ssim", "lpips", "de00")}
        feats, real, grays, t_map = [], [], [], 0.0
        torch.manual_seed(0)
        for A, B, g in loader:
            A, B = A.to(device), B.to(device)
            if kind == "input":
                y = to01(A)
            elif kind == "ceiling":
                y = to01(obj.decode_B(obj.encode_B(B)))
            else:
                if steps is not None:
                    net.steps = steps
                z_A = obj.encode_A(A)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                t0 = time.time()
                z_B = net(z_A)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                t_map += time.time() - t0
                y = to01(obj.decode_B(z_B))
            t = to01(B)
            for k, v in metrics.per_image(y, t).items():
                per[k].append(v)
            feats.append(metrics.feats(y))
            if real_feats is None:
                real.append(metrics.feats(t))
                grays.append(g.bool())
        if real_feats is None:
            real_feats, gray_all = np.concatenate(real), torch.cat(grays)
        per = {k: torch.cat(v) for k, v in per.items()}
        f = np.concatenate(feats)
        kid_m, kid_s = kid_from_feats(f, real_feats)
        row = {"method": name, "PSNR↑": per["psnr"].mean().item(), "SSIM↑": per["ssim"].mean().item(),
               "LPIPS↓": per["lpips"].mean().item(), "FID↓": fid_from_feats(f, real_feats),
               "KID×1000↓": 1000 * kid_m, "KID std×1000": 1000 * kid_s, "ΔE00↓": per["de00"].mean().item()}
        for tag, m in (("gray", gray_all), ("colour", ~gray_all)):
            if m.any():
                for k, col in (("psnr", "PSNR"), ("ssim", "SSIM"), ("lpips", "LPIPS"), ("de00", "ΔE00")):
                    row[f"{col} {tag}"] = per[k][m].mean().item()
        if kind == "net":
            row["mapping params (M)"] = sum(p.numel() for p in net.parameters()) / 1e6
            row["mapping ms/img"] = 1000 * t_map / len(ds)
        rows.append(row)
        print(f"{name:34s} PSNR {row['PSNR↑']:.2f}  SSIM {row['SSIM↑']:.4f}  LPIPS {row['LPIPS↓']:.4f}  "
              f"FID {row['FID↓']:.2f}  KID×1000 {row['KID×1000↓']:.2f}±{row['KID std×1000']:.2f}  ΔE00 {row['ΔE00↓']:.2f}")

    df = pd.DataFrame(rows).set_index("method")
    if out_csv:
        df.to_csv(out_csv)
        print(f"saved {out_csv}")
    return df.round(4)

### `color_analysis.py`

In [ ]:
%%writefile /content/color_analysis.py
"""
Colour-drift analysis: Microsoft mapping vs RF vs ground truth.

All comparisons are done in CIELAB (D65), where directions are readable:
    L*  lightness         0 (black) .. 100 (white)
    a*  green (-) .. red (+)
    b*  blue  (-) .. yellow (+)
    C*  chroma  = sqrt(a*^2 + b*^2)   (saturation)
    h   hue angle = atan2(b*, a*) in degrees (0 = pinkish red, 90 = yellow,
        180 = cyan/green-blue, 270 = blue)

What is measured (output vs the clean ground truth, same pixels):
  1. Overall error: CIEDE2000 dE (median / mean / p95), lightness shift dL*,
     and the colour change split into its two parts:
       - cast = shift of NEUTRAL pixels (gray/white in the ground truth), the
         white-balance definition, which desaturation cannot fake;
       - fit  output_ab = s * true_ab + c  over all pixels: s = saturation
         scale, c = cast. (A plain mean da*/db* mixes both: desaturating mostly
         magenta images looks like a green/yellow "cast".)
     plus chroma ratio (saturation gain/loss),
     colourfulness (Hasler-Suesstrunk), local lightness-contrast ratio
     (fine detail such as stripes collapsing).
  2. Per hue (colour images): for ground-truth pixels of each hue, the mean
     signed hue shift, chroma ratio, lightness shift, dE, and where they end up.
  3. Hue transition matrix: ground-truth colour class -> output colour class
     ("neutral" + 12 hues). Off-diagonal mass = colours turning into others;
     the "neutral" row/column = tinting gray / desaturating colour.
  4. Per tone (shadows / midtones / highlights): da*, db*, dL*, dC*: casts that
     differ by brightness (e.g. blue shadows, yellow highlights).
  5. Gray images: false colour added to gray content (mean a*, b*, C*, and
     share of visibly tinted pixels).

The test set is built exactly like evaluate.py's (held-out training split or a
separate folder, fixed seed), so numbers line up with the metric tables.
"""
import math
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from online_dataset import list_clean_images, make_online_pair

HUE_NAMES = ["pink", "red", "orange", "yellow", "lime", "green",
             "teal", "cyan", "sky", "blue", "violet", "magenta"]   # 30-degree bins of Lab hue, approximate
CLASSES = ["neutral"] + HUE_NAMES
TONES = [("shadows", 0, 33), ("midtones", 33, 66), ("highlights", 66, 101)]


# --------------------------------------------------------------------------- #
# Test set (same construction as evaluate.py)
# --------------------------------------------------------------------------- #
def build_test_set(clean_dir=None, test_clean_dir=None, n=500, grayscale_prob=0.5, seed=0, val_fraction=0.05):
    """A, B (N,3,256,256) in [-1,1] and gray (N,) bool. seed / val_fraction must
    match training's make_online_dataloaders call for a held-out split."""
    if test_clean_dir:
        files = list_clean_images(test_clean_dir)
    else:
        shuffled = list_clean_images(clean_dir)
        random.Random(seed).shuffle(shuffled)
        files = shuffled[:max(1, int(len(shuffled) * val_fraction))]
    state = random.getstate(), np.random.get_state()
    random.seed(seed + 123)
    np.random.seed(seed + 123)
    A, B, i = [], [], 0
    while len(A) < n and i < 20 * n:
        f = files[i % len(files)]
        i += 1
        try:
            a, b = make_online_pair(f, grayscale_prob)
        except Exception as e:
            print(f"[test set] skipping unreadable {f}: {e}")
            continue
        A.append(a)
        B.append(b)
    random.setstate(state[0])
    np.random.set_state(state[1])
    A, B = torch.stack(A), torch.stack(B)
    gray = (B - B.mean(1, keepdim=True)).abs().flatten(1).amax(1) < 1e-3
    print(f"test set: {len(A)} pairs from {len(files)} source images "
          f"({int(gray.sum())} gray, {int((~gray).sum())} colour)")
    return A, B, gray


# --------------------------------------------------------------------------- #
# Colour science (torch, on GPU)
# --------------------------------------------------------------------------- #
_M_RGB2XYZ = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                           [0.2126729, 0.7151522, 0.0721750],
                           [0.0193339, 0.1191920, 0.9503041]])
_WHITE_D65 = torch.tensor([0.95047, 1.0, 1.08883])


def srgb_to_lab(rgb):
    """(N,3,H,W) sRGB in [0,1] -> (N,3,H,W) CIELAB (D65, 2 deg), like skimage.color.rgb2lab."""
    rgb = rgb.clamp(0, 1)
    lin = torch.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    xyz = torch.einsum("ij,njhw->nihw", _M_RGB2XYZ.to(rgb), lin) / _WHITE_D65.to(rgb)[None, :, None, None]
    eps = (6 / 29) ** 3
    f = torch.where(xyz > eps, xyz.clamp_min(1e-12) ** (1 / 3), xyz / (3 * (6 / 29) ** 2) + 4 / 29)
    L = 116 * f[:, 1] - 16
    a = 500 * (f[:, 0] - f[:, 1])
    b = 200 * (f[:, 1] - f[:, 2])
    return torch.stack([L, a, b], 1)


def ciede2000(lab1, lab2):
    """Per-pixel CIEDE2000 between (N,3,H,W) Lab tensors -> (N,H,W). (Sharma et al. 2005)"""
    L1, a1, b1 = lab1[:, 0], lab1[:, 1], lab1[:, 2]
    L2, a2, b2 = lab2[:, 0], lab2[:, 1], lab2[:, 2]
    C1 = torch.sqrt(a1 ** 2 + b1 ** 2)
    C2 = torch.sqrt(a2 ** 2 + b2 ** 2)
    Cb7 = ((C1 + C2) / 2) ** 7
    G = 0.5 * (1 - torch.sqrt(Cb7 / (Cb7 + 25.0 ** 7)))
    a1p, a2p = (1 + G) * a1, (1 + G) * a2
    C1p = torch.sqrt(a1p ** 2 + b1 ** 2)
    C2p = torch.sqrt(a2p ** 2 + b2 ** 2)
    h1p = torch.rad2deg(torch.atan2(b1, a1p)) % 360
    h2p = torch.rad2deg(torch.atan2(b2, a2p)) % 360
    zero = (C1p * C2p) == 0

    dLp = L2 - L1
    dCp = C2p - C1p
    dh = h2p - h1p
    dh = torch.where(dh > 180, dh - 360, torch.where(dh < -180, dh + 360, dh))
    dh = torch.where(zero, torch.zeros_like(dh), dh)
    dHp = 2 * torch.sqrt(C1p * C2p) * torch.sin(torch.deg2rad(dh / 2))

    Lbp = (L1 + L2) / 2
    Cbp = (C1p + C2p) / 2
    hsum = h1p + h2p
    hbp = torch.where((h1p - h2p).abs() <= 180, hsum / 2,
                      torch.where(hsum < 360, (hsum + 360) / 2, (hsum - 360) / 2))
    hbp = torch.where(zero, hsum, hbp)
    T = (1 - 0.17 * torch.cos(torch.deg2rad(hbp - 30)) + 0.24 * torch.cos(torch.deg2rad(2 * hbp))
         + 0.32 * torch.cos(torch.deg2rad(3 * hbp + 6)) - 0.20 * torch.cos(torch.deg2rad(4 * hbp - 63)))
    dtheta = 30 * torch.exp(-(((hbp - 275) / 25) ** 2))
    Cbp7 = Cbp ** 7
    RC = 2 * torch.sqrt(Cbp7 / (Cbp7 + 25.0 ** 7))
    SL = 1 + 0.015 * (Lbp - 50) ** 2 / torch.sqrt(20 + (Lbp - 50) ** 2)
    SC = 1 + 0.045 * Cbp
    SH = 1 + 0.015 * Cbp * T
    RT = -torch.sin(torch.deg2rad(2 * dtheta)) * RC
    return torch.sqrt(((dLp / SL) ** 2 + (dCp / SC) ** 2 + (dHp / SH) ** 2
                       + RT * (dCp / SC) * (dHp / SH)).clamp_min(0))


def colourfulness(rgb):
    """(N,3,H,W) in [0,1] -> (N,) Hasler & Suesstrunk colourfulness (0-255 scale)."""
    x = rgb * 255
    rg = x[:, 0] - x[:, 1]
    yb = 0.5 * (x[:, 0] + x[:, 1]) - x[:, 2]
    std = torch.sqrt(rg.flatten(1).var(1) + yb.flatten(1).var(1))
    mean = torch.sqrt(rg.flatten(1).mean(1) ** 2 + yb.flatten(1).mean(1) ** 2)
    return std + 0.3 * mean


def local_contrast(L, k=8):
    """(N,H,W) lightness -> (N,) mean std of L* inside k x k patches (fine-detail contrast)."""
    L = L[:, None]
    m = F.avg_pool2d(L, k)
    v = (F.avg_pool2d(L * L, k) - m * m).clamp_min(0)
    return v.sqrt().flatten(1).mean(1)


def color_class(lab, chroma_thresh):
    """(N,3,H,W) Lab -> (N,H,W) long: 0 = neutral (C* < thresh), 1..12 = 30-degree hue bins."""
    C = torch.sqrt(lab[:, 1] ** 2 + lab[:, 2] ** 2)
    h = torch.rad2deg(torch.atan2(lab[:, 2], lab[:, 1])) % 360
    hb = (h // 30).long().clamp(0, 11) + 1
    return torch.where(C < chroma_thresh, torch.zeros_like(hb), hb)


# --------------------------------------------------------------------------- #
# Statistics accumulation
# --------------------------------------------------------------------------- #
_DE_BINS = torch.linspace(0, 60, 601)   # dE histogram for median / p95


class _Acc:
    """Accumulates colour statistics for one method across batches."""

    def __init__(self, device):
        z = lambda *s: torch.zeros(*s, dtype=torch.float64, device=device)
        self.de_hist = {"colour": z(600), "gray": z(600)}
        self.img = {"colour": {k: [] for k in ("dE", "dL", "da", "db", "chroma_ratio", "cf_out", "cf_gt", "contrast_ratio")},
                    "gray": {k: [] for k in ("dE", "dL", "a", "b", "C", "tinted", "contrast_ratio")}}
        self.hue = {k: z(12) for k in ("n", "n_both", "dh", "dh_abs", "C_gt", "C_out", "dL", "dE",
                                       "a_gt", "b_gt", "a_out", "b_out", "stay")}
        self.trans = z(13, 13)
        self.gray_trans = z(13)
        self.tone = {k: z(3) for k in ("n", "dL", "dC", "n_neu", "da_neu", "db_neu")}
        self.fit = z(7)       # N, sum a_g, sum b_g, sum a_o, sum b_o, sum |g|^2, sum o.g  (colour images)
        self.neutral = z(3)   # n, sum da, sum db over ground-truth-neutral pixels (colour images)
        self.hist = {k: z(100) for k in ("L_gt", "L_out", "a_gt", "a_out", "b_gt", "b_out")}

    def add(self, lab_o, lab_g, rgb_o, rgb_g, is_gray, chroma_thresh, tint_thresh):
        dE = ciede2000(lab_g, lab_o)
        d = lab_o - lab_g
        C_o = torch.sqrt(lab_o[:, 1] ** 2 + lab_o[:, 2] ** 2)
        C_g = torch.sqrt(lab_g[:, 1] ** 2 + lab_g[:, 2] ** 2)
        contrast = local_contrast(lab_o[:, 0]) / local_contrast(lab_g[:, 0]).clamp_min(1e-3)

        for tag, m in (("colour", ~is_gray), ("gray", is_gray)):
            if not m.any():
                continue
            self.de_hist[tag] += torch.histc(dE[m].double(), 600, 0, 60)
            rec = self.img[tag]
            rec["dE"].append(dE[m].flatten(1).mean(1).cpu())
            rec["dL"].append(d[m, 0].flatten(1).mean(1).cpu())
            rec["contrast_ratio"].append(contrast[m].cpu())
            if tag == "colour":
                rec["da"].append(d[m, 1].flatten(1).mean(1).cpu())
                rec["db"].append(d[m, 2].flatten(1).mean(1).cpu())
                rec["chroma_ratio"].append((C_o[m].flatten(1).mean(1) / C_g[m].flatten(1).mean(1).clamp_min(1e-3)).cpu())
                rec["cf_out"].append(colourfulness(rgb_o[m]).cpu())
                rec["cf_gt"].append(colourfulness(rgb_g[m]).cpu())
            else:
                rec["a"].append(lab_o[m, 1].flatten(1).mean(1).cpu())
                rec["b"].append(lab_o[m, 2].flatten(1).mean(1).cpu())
                rec["C"].append(C_o[m].flatten(1).mean(1).cpu())
                rec["tinted"].append((C_o[m] > tint_thresh).float().flatten(1).mean(1).cpu())
                cls_o = color_class(lab_o[m], chroma_thresh)
                self.gray_trans += torch.bincount(cls_o.flatten(), minlength=13).double()

        if (~is_gray).any():
            m = ~is_gray
            lo, lg = lab_o[m], lab_g[m]
            cg, co = color_class(lg, chroma_thresh), color_class(lo, chroma_thresh)
            self.trans += torch.bincount((cg * 13 + co).flatten(), minlength=169).double().view(13, 13)

            Cg, Co = C_g[m].flatten(), C_o[m].flatten()
            hg = torch.rad2deg(torch.atan2(lg[:, 2], lg[:, 1])).flatten() % 360
            ho = torch.rad2deg(torch.atan2(lo[:, 2], lo[:, 1])).flatten() % 360
            dh = (ho - hg + 180) % 360 - 180          # signed, + = counter-clockwise (red -> yellow -> green -> blue)
            bin_g = (hg // 30).long().clamp(0, 11)
            chrom = Cg >= chroma_thresh                # hue only meaningful for chromatic ground truth
            both = chrom & (Co >= chroma_thresh)
            flat = lambda x: x.flatten()
            dEm, dLm = flat(dE[m]), flat(d[m, 0])
            vals = {"n": chrom.double(), "n_both": both.double(),
                    "dh": torch.where(both, dh, torch.zeros_like(dh)).double(),
                    "dh_abs": torch.where(both, dh.abs(), torch.zeros_like(dh)).double(),
                    "C_gt": torch.where(chrom, Cg, torch.zeros_like(Cg)).double(),
                    "C_out": torch.where(chrom, Co, torch.zeros_like(Co)).double(),
                    "dL": torch.where(chrom, dLm, torch.zeros_like(dLm)).double(),
                    "dE": torch.where(chrom, dEm, torch.zeros_like(dEm)).double(),
                    "a_gt": torch.where(chrom, flat(lg[:, 1]), torch.zeros_like(Cg)).double(),
                    "b_gt": torch.where(chrom, flat(lg[:, 2]), torch.zeros_like(Cg)).double(),
                    "a_out": torch.where(chrom, flat(lo[:, 1]), torch.zeros_like(Cg)).double(),
                    "b_out": torch.where(chrom, flat(lo[:, 2]), torch.zeros_like(Cg)).double(),
                    "stay": (chrom & (flat(co) == bin_g + 1)).double()}
            for k, v in vals.items():
                self.hue[k] += torch.bincount(bin_g, weights=v, minlength=12)

            ag, bg, ao, bo = (x.double().flatten() for x in (lg[:, 1], lg[:, 2], lo[:, 1], lo[:, 2]))
            self.fit += torch.stack([torch.tensor(float(ag.numel()), device=ag.device, dtype=torch.float64),
                                     ag.sum(), bg.sum(), ao.sum(), bo.sum(),
                                     (ag * ag + bg * bg).sum(), (ao * ag + bo * bg).sum()])
            neu = C_g[m] < chroma_thresh
            if neu.any():
                self.neutral += torch.stack([neu.sum().double(), (lo[:, 1] - lg[:, 1])[neu].double().sum(),
                                             (lo[:, 2] - lg[:, 2])[neu].double().sum()])

            Lg = lg[:, 0]
            for ti, (_, lo_, hi_) in enumerate(TONES):
                tm = (Lg >= lo_) & (Lg < hi_)
                if tm.any():
                    self.tone["n"][ti] += tm.sum()
                    self.tone["dL"][ti] += (lo[:, 0] - lg[:, 0])[tm].double().sum()
                    self.tone["dC"][ti] += (C_o[m] - C_g[m])[tm].double().sum()
                    tn = tm & neu
                    if tn.any():
                        self.tone["n_neu"][ti] += tn.sum()
                        self.tone["da_neu"][ti] += (lo[:, 1] - lg[:, 1])[tn].double().sum()
                        self.tone["db_neu"][ti] += (lo[:, 2] - lg[:, 2])[tn].double().sum()

            for ch, idx, rng in (("L", 0, (0, 100)), ("a", 1, (-80, 80)), ("b", 2, (-80, 80))):
                self.hist[f"{ch}_gt"] += torch.histc(lg[:, idx].double(), 100, *rng)
                self.hist[f"{ch}_out"] += torch.histc(lo[:, idx].double(), 100, *rng)

    def finish(self):
        out = {"de_hist": {k: v.cpu() for k, v in self.de_hist.items()},
               "img": {t: {k: torch.cat(v) if v else torch.empty(0) for k, v in d.items()} for t, d in self.img.items()},
               "hue": {k: v.cpu() for k, v in self.hue.items()},
               "trans": self.trans.cpu(), "gray_trans": self.gray_trans.cpu(),
               "tone": {k: v.cpu() for k, v in self.tone.items()},
               "fit": self.fit.cpu(), "neutral": self.neutral.cpu(),
               "hist": {k: v.cpu() for k, v in self.hist.items()}}
        return out


def _to01(y):
    return ((y.float() + 1) / 2).clamp(0, 1)


@torch.no_grad()
def analyze(obj, rfs, A, B, gray, rf_steps=(8,), batch_size=16, include_input=True,
            chroma_thresh=8.0, tint_thresh=5.0):
    """obj: OldPhotoModel (quality); rfs: {"name": RFMapping}. Returns {method: stats}.
    chroma_thresh: C* below which a pixel counts as neutral (gray).
    tint_thresh:   C* above which a pixel of a gray image counts as visibly tinted."""
    device = obj.device
    methods = {}
    if include_input:
        methods["degraded input"] = (None, None)
    methods["Microsoft mapping"] = (obj._core.mapping_net, None)
    for name, rf in rfs.items():
        for s in rf_steps:            # s=None: a direct mapping (no sampler), labelled by its name only
            methods[name if s is None else f"{name} ({s}-step)"] = (rf, s)

    stats = {}
    n = len(A)
    for name, (net, steps) in methods.items():
        acc = _Acc(device)
        for i in range(0, n, batch_size):
            a = A[i:i + batch_size].to(device)
            rgb_g = _to01(B[i:i + batch_size].to(device))
            if net is None:
                rgb_o = _to01(a)
            else:
                if steps is not None:
                    net.steps = steps
                rgb_o = _to01(obj.decode_B(net(obj.encode_A(a))))
            acc.add(srgb_to_lab(rgb_o), srgb_to_lab(rgb_g), rgb_o, rgb_g,
                    gray[i:i + batch_size].to(device), chroma_thresh, tint_thresh)
        stats[name] = acc.finish()
        print(f"analyzed {name}")
    stats["_config"] = {"chroma_thresh": chroma_thresh, "tint_thresh": tint_thresh}
    return stats


# --------------------------------------------------------------------------- #
# Tables
# --------------------------------------------------------------------------- #
def _methods(stats):
    return [k for k in stats if not k.startswith("_")]


def _hist_quantile(hist, q):
    c = hist.cumsum(0)
    if c[-1] == 0:
        return float("nan")
    idx = int(torch.searchsorted(c, q * c[-1]))
    return float(_DE_BINS[min(idx + 1, 600)])


def _cast_name(da, db, tol=0.5):
    parts = []
    if db < -tol:
        parts.append("blue")
    elif db > tol:
        parts.append("yellow")
    if da > tol:
        parts.append("red/magenta")
    elif da < -tol:
        parts.append("green")
    return "+".join(parts) if parts else "none"


def _fit_sat_cast(fit):
    """Least squares output_ab = s * true_ab + c  ->  (s, c_a, c_b)."""
    N, sag, sbg, sao, sbo, sgg, sog = fit.tolist()
    if N < 1:
        return float("nan"), float("nan"), float("nan")
    s = (sog - (sao * sag + sbo * sbg) / N) / max(sgg - (sag ** 2 + sbg ** 2) / N, 1e-9)
    return s, (sao - s * sag) / N, (sbo - s * sbg) / N


MIN_NEUTRAL_PX = 2000   # fewer ground-truth-neutral pixels than this -> use the fitted cast instead


def _cast(s):
    """(cast_a, cast_b, source) for one method's colour images."""
    n, sa, sb = s["neutral"].tolist()
    if n >= MIN_NEUTRAL_PX:
        return sa / n, sb / n, "neutral px"
    _, ca, cb = _fit_sat_cast(s["fit"])
    return ca, cb, "fit"


def summary_tables(stats):
    """Returns (colour_df, gray_df): one row per method."""
    col_rows, gray_rows = [], []
    for name in _methods(stats):
        s = stats[name]
        c, g = s["img"]["colour"], s["img"]["gray"]
        if len(c["dE"]):
            cast_a, cast_b, src = _cast(s)
            sat, fit_a, fit_b = _fit_sat_cast(s["fit"])
            col_rows.append({
                "method": name,
                "dE00 median": _hist_quantile(s["de_hist"]["colour"], 0.5),
                "dE00 mean": c["dE"].mean().item(),
                "dE00 p95": _hist_quantile(s["de_hist"]["colour"], 0.95),
                "dL* (lightness)": c["dL"].mean().item(),
                "cast a* (+red/-green)": cast_a,
                "cast b* (+yellow/-blue)": cast_b,
                "cast": _cast_name(cast_a, cast_b),
                "cast measured on": src,
                "saturation scale s (fit)": sat,
                "cast a* (fit)": fit_a,
                "cast b* (fit)": fit_b,
                "mean da* (cast+sat mixed)": c["da"].mean().item(),
                "mean db* (cast+sat mixed)": c["db"].mean().item(),
                "chroma ratio (out/gt)": c["chroma_ratio"].mean().item(),
                "colourfulness out": c["cf_out"].mean().item(),
                "colourfulness gt": c["cf_gt"].mean().item(),
                "detail contrast ratio": c["contrast_ratio"].mean().item(),
                "mean |hue shift| deg": (s["hue"]["dh_abs"].sum() / s["hue"]["n_both"].sum().clamp_min(1)).item(),
            })
        if len(g["dE"]):
            a, b = g["a"].mean().item(), g["b"].mean().item()
            gray_rows.append({
                "method": name,
                "dE00 median": _hist_quantile(s["de_hist"]["gray"], 0.5),
                "dE00 mean": g["dE"].mean().item(),
                "dL* (lightness)": g["dL"].mean().item(),
                "false a* (+red/-green)": a,
                "false b* (+yellow/-blue)": b,
                "tint": _cast_name(a, b),
                "false chroma C*": g["C"].mean().item(),
                f"% pixels tinted (C*>{stats['_config']['tint_thresh']:g})": 100 * g["tinted"].mean().item(),
                "detail contrast ratio": g["contrast_ratio"].mean().item(),
            })
    to_df = lambda r: pd.DataFrame(r).set_index("method").round(3) if r else pd.DataFrame()
    return to_df(col_rows), to_df(gray_rows)


def hue_table(stats, method):
    """Per ground-truth hue (colour images): where that colour goes."""
    h = stats[method]["hue"]
    trans = stats[method]["trans"]
    rows = []
    total = h["n"].sum().clamp_min(1)
    for i, hn in enumerate(HUE_NAMES):
        n = h["n"][i]
        if n < 1:
            continue
        nb = h["n_both"][i].clamp_min(1)
        row_t = trans[i + 1]
        dest = row_t.clone()
        dest[i + 1] = 0
        top = int(dest.argmax())
        rows.append({
            "gt hue": f"{hn} ({30 * i}-{30 * i + 30} deg)",
            "% of colour px": (100 * n / total).item(),
            "hue shift deg (+ccw)": (h["dh"][i] / nb).item(),
            "chroma ratio": (h["C_out"][i] / h["C_gt"][i].clamp_min(1e-6)).item(),
            "dL*": (h["dL"][i] / n).item(),
            "dE00": (h["dE"][i] / n).item(),
            "% stay same hue": (100 * h["stay"][i] / n).item(),
            "% -> neutral": (100 * row_t[0] / row_t.sum().clamp_min(1)).item(),
            "most common drift": f"{CLASSES[top]} ({100 * dest[top] / row_t.sum().clamp_min(1):.0f}%)",
        })
    return pd.DataFrame(rows).set_index("gt hue").round(2)


def tone_table(stats):
    """Per lightness band (colour images): lightness and saturation shift over all
    pixels, and the cast measured on ground-truth-neutral pixels of that band."""
    rows = []
    for name in _methods(stats):
        t = stats[name]["tone"]
        for i, (tn, lo, hi) in enumerate(TONES):
            n = t["n"][i].clamp_min(1)
            nn_ = t["n_neu"][i]
            ok = bool(nn_ >= MIN_NEUTRAL_PX / 3)
            da = (t["da_neu"][i] / nn_).item() if ok else float("nan")
            db = (t["db_neu"][i] / nn_).item() if ok else float("nan")
            rows.append({"method": name, "tone": f"{tn} (L* {lo}-{min(hi, 100)})",
                         "dL*": (t["dL"][i] / n).item(), "dC* (saturation)": (t["dC"][i] / n).item(),
                         "cast da*": da, "cast db*": db,
                         "cast": _cast_name(da, db) if ok else "n/a (few neutral px)",
                         "neutral px": int(nn_)})
    return pd.DataFrame(rows).set_index(["method", "tone"]).round(2)


def transition_table(stats, method):
    """Row-normalized % : ground-truth colour class (rows) -> output colour class (cols)."""
    t = stats[method]["trans"]
    pct = 100 * t / t.sum(1, keepdim=True).clamp_min(1)
    df = pd.DataFrame(pct.numpy(), index=[f"gt {c}" for c in CLASSES], columns=[f"-> {c}" for c in CLASSES])
    return df[t.sum(1).numpy() > 0].round(1)


def describe(stats, min_share=3.0, min_shift=8.0, min_chroma_change=0.15):
    """Plain-language findings per method."""
    col, gray = summary_tables(stats)
    lines = []
    for name in _methods(stats):
        lines.append(f"\n=== {name} ===")
        if name in col.index:
            r = col.loc[name]
            lines.append(f"Colour images: median dE00 {r['dE00 median']:.1f} (p95 {r['dE00 p95']:.1f}); "
                         f"global cast: {r['cast']} (a* {r['cast a* (+red/-green)']:+.2f}, "
                         f"b* {r['cast b* (+yellow/-blue)']:+.2f}, on {r['cast measured on']}); "
                         f"lightness {r['dL* (lightness)']:+.2f} L*; saturation x{r['chroma ratio (out/gt)']:.2f} "
                         f"(fit s={r['saturation scale s (fit)']:.2f}); "
                         f"fine-detail contrast x{r['detail contrast ratio']:.2f}.")
            ht = hue_table(stats, name)
            for hue, h in ht.iterrows():
                if h["% of colour px"] < min_share:
                    continue
                notes = []
                shift = h["hue shift deg (+ccw)"]
                if abs(shift) >= min_shift:
                    i = HUE_NAMES.index(hue.split()[0])
                    towards = HUE_NAMES[(i + 1) % 12] if shift > 0 else HUE_NAMES[i - 1]  # + = ccw on the hue wheel
                    notes.append(f"hue {shift:+.0f} deg (towards {towards})")
                if abs(h["chroma ratio"] - 1) >= min_chroma_change:
                    notes.append(("more" if h["chroma ratio"] > 1 else "less") + f" saturated (x{h['chroma ratio']:.2f})")
                if h["% -> neutral"] >= 20:
                    notes.append(f"{h['% -> neutral']:.0f}% washed out to neutral")
                if notes:
                    lines.append(f"  - {hue}: " + ", ".join(notes) + f"; most common drift: {h['most common drift']}")
        if name in gray.index:
            r = gray.loc[name]
            tint_col = [c for c in gray.columns if c.startswith("% pixels tinted")][0]
            lines.append(f"Gray images: false tint {r['tint']} (a* {r['false a* (+red/-green)']:+.2f}, "
                         f"b* {r['false b* (+yellow/-blue)']:+.2f}, C* {r['false chroma C*']:.2f}); "
                         f"{r[tint_col]:.1f}% of pixels visibly tinted; lightness {r['dL* (lightness)']:+.2f} L*.")
    tone = tone_table(stats)
    lines.append("\nTone-dependent casts (colour images):")
    for (name, tn), r in tone.iterrows():
        if r["cast"] != "none" and not r["cast"].startswith("n/a"):
            lines.append(f"  {name} | {tn}: {r['cast']} (da* {r['cast da*']:+.2f}, db* {r['cast db*']:+.2f}), "
                         f"dL* {r['dL*']:+.2f}")
    return "\n".join(lines)


# --------------------------------------------------------------------------- #
# Plots
# --------------------------------------------------------------------------- #
def _lab_to_rgb_tuple(L, a, b):
    """Single Lab colour -> matplotlib RGB tuple (for drawing hue swatches)."""
    fy = (L + 16) / 116
    fx, fz = fy + a / 500, fy - b / 200
    inv = lambda f: f ** 3 if f ** 3 > (6 / 29) ** 3 else 3 * (6 / 29) ** 2 * (f - 4 / 29)
    xyz = np.array([inv(fx), inv(fy), inv(fz)]) * _WHITE_D65.numpy()
    lin = np.linalg.solve(_M_RGB2XYZ.numpy(), xyz)
    rgb = np.where(lin <= 0.0031308, 12.92 * lin, 1.055 * np.clip(lin, 0, None) ** (1 / 2.4) - 0.055)
    return tuple(np.clip(rgb, 0, 1))


def _savefig(plt, path):
    """Save the current figure, creating its folder first (e.g. a new Drive folder)."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    plt.savefig(path, dpi=150)


def plot_ab_drift(stats, save=None):
    """a*b* plane: arrow from each ground-truth hue's mean colour to where the method puts it."""
    import matplotlib.pyplot as plt
    names = [n for n in _methods(stats) if n != "degraded input"]
    fig, axes = plt.subplots(1, len(names), figsize=(5 * len(names), 5), squeeze=False)
    lim = 30.0   # common axis range covering every dot and arrow tip
    for name in names:
        h = stats[name]["hue"]
        n = h["n"].clamp_min(1)
        for k in ("a_gt", "b_gt", "a_out", "b_out"):
            lim = max(lim, float((h[k] / n)[h["n"] >= 1].abs().max()) * 1.15 if (h["n"] >= 1).any() else lim)
    for ax, name in zip(axes[0], names):
        h = stats[name]["hue"]
        for i, hn in enumerate(HUE_NAMES):
            n = h["n"][i]
            if n < 1:
                continue
            ag, bg = (h["a_gt"][i] / n).item(), (h["b_gt"][i] / n).item()
            ao, bo = (h["a_out"][i] / n).item(), (h["b_out"][i] / n).item()
            colour = _lab_to_rgb_tuple(60, ag, bg)
            ax.scatter([ag], [bg], s=60 + 400 * (n / h["n"].sum()).item(), color=colour, edgecolors="k", zorder=3)
            ax.annotate("", xy=(ao, bo), xytext=(ag, bg),
                        arrowprops=dict(arrowstyle="->", color="k", lw=1.5), zorder=4)
            ax.text(ag, bg, f" {hn}", fontsize=8, va="bottom")
        g = stats[name]["img"]["gray"]
        if len(g["a"]):
            ax.scatter([g["a"].mean()], [g["b"].mean()], marker="x", s=120, color="k", zorder=5,
                       label="gray images: mean false colour")
            ax.legend(loc="lower left", fontsize=8)
        ax.axhline(0, color="0.7", lw=0.8)
        ax.axvline(0, color="0.7", lw=0.8)
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_xlabel("a*  (green -  /  + red)")
        ax.set_ylabel("b*  (blue -  /  + yellow)")
        ax.set_title(f"{name}\ndot = true mean colour per hue, arrow = output")
        ax.set_aspect("equal")
    plt.tight_layout()
    if save:
        _savefig(plt, save)
    plt.show()


def plot_transitions(stats, save=None):
    """Heatmap of ground-truth colour class -> output colour class (% of each row)."""
    import matplotlib.pyplot as plt
    names = [n for n in _methods(stats) if n != "degraded input"]
    fig, axes = plt.subplots(1, len(names), figsize=(6.5 * len(names), 6), squeeze=False)
    for ax, name in zip(axes[0], names):
        t = stats[name]["trans"]
        keep = (t.sum(1) > 0).numpy()
        pct = (100 * t / t.sum(1, keepdim=True).clamp_min(1)).numpy()[keep]
        im = ax.imshow(pct, cmap="viridis", vmin=0, vmax=100)
        ax.set_xticks(range(13))
        ax.set_xticklabels(CLASSES, rotation=60, fontsize=8)
        ax.set_yticks(range(int(keep.sum())))
        ax.set_yticklabels([c for c, k in zip(CLASSES, keep) if k], fontsize=8)
        for r in range(pct.shape[0]):
            for c in range(13):
                if pct[r, c] >= 5:
                    ax.text(c, r, f"{pct[r, c]:.0f}", ha="center", va="center", fontsize=7,
                            color="w" if pct[r, c] < 60 else "k")
        ax.set_xlabel("output colour")
        ax.set_ylabel("ground-truth colour")
        ax.set_title(f"{name}\n% of each true colour ending up as ...")
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    if save:
        _savefig(plt, save)
    plt.show()


def plot_tone_casts(stats, save=None):
    """da*, db*, dL* per lightness band, per method."""
    import matplotlib.pyplot as plt
    tone = tone_table(stats)
    names = _methods(stats)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    width = 0.8 / len(names)
    for ax, col, lab in zip(axes, ("cast da*", "cast db*", "dL*"),
                            ("cast da* on neutral px  (+ red / - green)", "cast db* on neutral px  (+ yellow / - blue)",
                             "dL*  (+ brighter / - darker)")):
        for j, name in enumerate(names):
            vals = [tone.loc[(name, f"{tn} (L* {lo}-{min(hi, 100)})"), col] for tn, lo, hi in TONES]
            ax.bar(np.arange(3) + j * width, vals, width, label=name)
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(np.arange(3) + width * (len(names) - 1) / 2)
        ax.set_xticklabels([t[0] for t in TONES])
        ax.set_ylabel(lab)
    axes[0].legend(fontsize=7)
    fig.suptitle("Colour shift by tone (colour images, output - ground truth)")
    plt.tight_layout()
    if save:
        _savefig(plt, save)
    plt.show()


def plot_histograms(stats, save=None):
    """L*, a*, b* distributions of outputs vs ground truth (colour images)."""
    import matplotlib.pyplot as plt
    names = _methods(stats)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, ch, rng, lab in zip(axes, ("L", "a", "b"), ((0, 100), (-80, 80), (-80, 80)),
                                ("L* (lightness)", "a* (- green / + red)", "b* (- blue / + yellow)")):
        x = np.linspace(rng[0], rng[1], 100)
        gt = stats[names[0]]["hist"][f"{ch}_gt"].numpy()
        ax.plot(x, gt / gt.sum(), "k", lw=2.5, label="ground truth")
        for name in names:
            h = stats[name]["hist"][f"{ch}_out"].numpy()
            ax.plot(x, h / h.sum(), lw=1.2, label=name)
        ax.set_xlabel(lab)
        if ch != "L":
            ax.set_yscale("log")
    axes[0].legend(fontsize=7)
    fig.suptitle("Pixel distributions (colour images)")
    plt.tight_layout()
    if save:
        _savefig(plt, save)
    plt.show()


def save_tables(stats, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    col, gray = summary_tables(stats)
    col.to_csv(os.path.join(out_dir, "colour_summary.csv"))
    gray.to_csv(os.path.join(out_dir, "gray_summary.csv"))
    tone_table(stats).to_csv(os.path.join(out_dir, "tone_casts.csv"))
    for name in _methods(stats):
        safe = "".join(ch if ch.isalnum() else "_" for ch in name)
        hue_table(stats, name).to_csv(os.path.join(out_dir, f"hue_{safe}.csv"))
        transition_table(stats, name).to_csv(os.path.join(out_dir, f"transitions_{safe}.csv"))
    with open(os.path.join(out_dir, "findings.txt"), "w") as f:
        f.write(describe(stats))
    print(f"saved tables + findings.txt to {out_dir}")

## Configuration

- **`OURS`:** the final model. Either a `DirectMapping` checkpoint, or a velocity-network checkpoint trained at t = 0, which is folded automatically.
- **`OURS_RUN2`:** an optional second training run, to show the run-to-run spread.
- **`FLOW`:** the optional bridge-flow checkpoint for the ablation, scored at `FLOW_STEPS`.

In [ ]:
ROOT = "/content/drive/MyDrive/lightweight_old_photos"      # project folder on Google Drive
OURS = f"{ROOT}/checkpoints/direct/final.pt"
OURS_RUN2 = None                                   # e.g. f"{ROOT}/checkpoints/direct_seed1/final.pt"
FLOW = f"{ROOT}/checkpoints/bridge_flow/final.pt"  # set None to skip the ablation
FLOW_STEPS = (1, 4, 8)
BENCH_DIR = f"{ROOT}/benchmarks/coco_5000"
CLEAN_DIR = f"{ROOT}/data/voc_clean"                # held-out split = colour test set
REAL_DIR = "/content/Bringing-Old-Photos-Back-to-Life/test_images/old"
OUT_DIR = f"{ROOT}/results"
MAX_SIDE = 1024
os.makedirs(OUT_DIR, exist_ok=True)
summary = {}

## Load the models; folding check

In [ ]:
import glob, json, time
import numpy as np
import pandas as pd
import torch
from PIL import Image
from direct_model import DirectMapping, fold_rf
from old_photo_model import OldPhotoModel
from rf_model import RFMapping

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")
n_params = lambda m: sum(p.numel() for p in m.parameters())
if "obj" not in globals():
    obj = OldPhotoModel(mode="quality", gpu=0)
dev = obj.device
ms_net, ms_mapping, ms_uses_mask = obj._core.mapping_net, obj.mapping, obj.uses_mask


def load_mapping(path):
    """-> (model, step, kind): a DirectMapping (velocity nets trained at t = 0 are folded) or a flow RFMapping."""
    ck = torch.load(path, map_location=dev)
    a = ck["arch"]
    if a.get("type") == "direct":
        m = DirectMapping(a["latent_ch"], a["width"], a["n_blocks"]).to(dev)
        m.load_state_dict(ck["ema"])
        return m.eval(), ck["step"], "direct"
    m = RFMapping(a["latent_ch"], a["width"], a["n_blocks"], sigma=a.get("sigma", 0.0)).to(dev)
    m.load_state_dict(ck["ema"])
    if a.get("t_mode") == "direct":
        return fold_rf(m.eval()), ck["step"], "direct (folded from a t = 0 velocity network)"
    return m.eval(), ck["step"], "flow"


# folding check on a random velocity network: the t = 0 output equals the folded DirectMapping
torch.manual_seed(0)
probe = RFMapping(64, 256, 9).to(dev).eval()
for p in probe.parameters():
    torch.nn.init.normal_(p, std=0.02)
z = torch.randn(2, 64, 64, 64, device=dev)
with torch.no_grad():
    rel = ((probe.sample(z, steps=1) - fold_rf(probe)(z)).abs().max() / probe.sample(z, steps=1).abs().max()).item()
summary["folding_check"] = {"velocity_net_params_M": n_params(probe) / 1e6,
                            "direct_params_M": n_params(fold_rf(probe)) / 1e6, "max_relative_difference": rel}
print(f"FOLDING CHECK: velocity net {n_params(probe) / 1e6:.2f}M at t = 0 == DirectMapping "
      f"{n_params(fold_rf(probe)) / 1e6:.2f}M (max relative difference {rel:.1e})")

ours, ours_step, kind = load_mapping(OURS)
print(f"ours: {kind}, step {ours_step}, {n_params(ours) / 1e6:.2f}M params (BOPBTL {n_params(ms_net) / 1e6:.2f}M)")
run2 = load_mapping(OURS_RUN2) if OURS_RUN2 and os.path.exists(OURS_RUN2) else None
flow = load_mapping(FLOW) if FLOW and os.path.exists(FLOW) else None

## COCO benchmark

**Tables:**
1. Main results.
2. **Share of the achievable gain** over the degraded input: of BOPBTL's gain, and of the gain up to the VAE ceiling. PSNR and ΔE00 are omitted, because the input already beats every mapping on them.
3. Gray / colour split.
4. **Ablation:** direct vs flow at 1, 4 and 8 steps.

In [ ]:
from benchmark import evaluate_benchmark

CACHE = f"{OUT_DIR}/coco_all.csv"
REF = ["degraded input", "Microsoft mapping", "VAE B ceiling"]
want = {"Ours": (ours, None, f"{OURS}@{ours_step}")}
if run2:
    want["Ours, run 2"] = (run2[0], None, f"{OURS_RUN2}@{run2[1]}")
if flow:
    for k in FLOW_STEPS:
        want[f"Bridge flow ({k}-step)"] = (flow[0], k, f"{FLOW}@{flow[1]}")

cache = pd.read_csv(CACHE, index_col="method") if os.path.exists(CACHE) else pd.DataFrame(columns=["checkpoint"])
new = []
if any(r not in cache.index for r in REF):
    new.append(evaluate_benchmark(obj, {}, BENCH_DIR, include_input=True, include_ceiling=True))
for row, (net, k, ck) in want.items():
    if row not in cache.index or cache.loc[row, "checkpoint"] != ck:
        base = row if k is None else row.rsplit(" (", 1)[0]
        new.append(evaluate_benchmark(obj, {base: net}, BENCH_DIR, rf_steps=(k,), include_input=False,
                                      include_ceiling=False, include_microsoft=False).assign(checkpoint=ck))
if new:
    new = pd.concat(new)
    cache = pd.concat([cache.drop(index=[r for r in new.index if r in cache.index]), new])
res = cache.loc[[r for r in ["degraded input", "Microsoft mapping", *want, "VAE B ceiling"] if r in cache.index]]
res.index.name = "method"
res.to_csv(CACHE)

M = ["PSNR↑", "SSIM↑", "LPIPS↓", "FID↓", "KID×1000↓", "KID std×1000", "ΔE00↓"]
main_rows = ["degraded input", "Microsoft mapping", "Ours"] + (["Ours, run 2"] if run2 else []) + ["VAE B ceiling"]
print("1. MAIN RESULTS"); display(res.loc[main_rows, M].round(4))
inp, ms, ceil = res.loc["degraded input"], res.loc["Microsoft mapping"], res.loc["VAE B ceiling"]
G = ["SSIM↑", "LPIPS↓", "FID↓", "KID×1000↓"]
share = {}
for r in [r for r in main_rows if r.startswith("Ours")]:
    share[f"{r}: of BOPBTL's gain"] = {c: (res.loc[r, c] - inp[c]) / (ms[c] - inp[c]) for c in G}
for r in ["Microsoft mapping"] + [r for r in main_rows if r.startswith("Ours")]:
    share[f"{r}: of the ceiling's gain"] = {c: (res.loc[r, c] - inp[c]) / (ceil[c] - inp[c]) for c in G}
share = pd.DataFrame(share).T
print("2. SHARE OF THE ACHIEVABLE GAIN (%)"); display((100 * share).round(0))
split = [c for c in res.columns if c.endswith(" gray") or c.endswith(" colour")]
print("3. GRAY / COLOUR SPLIT"); display(res.loc[main_rows[1:-1], split].round(4))
abl = res.loc[[r for r in want], M]
print("4. ABLATION: direct vs bridge flow (same network, losses, data, budget)"); display(abl.round(4))
summary["coco"] = {"results": res[M].round(4).to_dict(orient="index"), "share_pct": (100 * share).round(1).to_dict(orient="index"),
                   "split": res.loc[main_rows[1:-1], split].round(4).to_dict(orient="index")}

## Efficiency (Tesla T4 in the paper; 10 warm-up runs, mean of 50)

In [ ]:
from torch.utils.flop_counter import FlopCounterMode


@torch.no_grad()
def time_ms(fn, warmup=10, iters=50):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return 1000 * (time.time() - t0) / iters


@torch.no_grad()
def gflops(fn):
    with FlopCounterMode(display=False) as fc:
        fn()
    return fc.get_total_flops() / 1e9


nets = {"BOPBTL": (ms_net, lambda z: ms_net(z)), "Ours": (ours, lambda z: ours(z))}
if flow:
    nets["Bridge flow, 1 step"] = (flow[0], lambda z: flow[0].sample(z, steps=1))
    nets["Bridge flow, 4 steps"] = (flow[0], lambda z: flow[0].sample(z, steps=4))
x1 = torch.rand(1, 3, 256, 256, device=dev) * 2 - 1
with torch.no_grad():
    z1 = obj.encode_A(x1)
rows = {}
for name, (m, f) in nets.items():
    r = {"params_M": n_params(m) / 1e6, "mapping_gflops_256": gflops(lambda: f(z1))}
    for bs in (16, 1):
        x = torch.rand(bs, 3, 256, 256, device=dev) * 2 - 1
        with torch.no_grad():
            z = obj.encode_A(x)
        r[f"mapping_ms_bs{bs}"] = time_ms(lambda: f(z)) / bs
        r[f"pipeline_ms_bs{bs}"] = time_ms(lambda: obj.decode_B(f(obj.encode_A(x)))) / bs
    rows[name] = r
eff = pd.DataFrame(rows).T
print(torch.cuda.get_device_name(0)); display(eff.round(2))
eff.to_csv(f"{OUT_DIR}/efficiency.csv")
summary["efficiency"] = {"gpu": torch.cuda.get_device_name(0), "rows": eff.round(3).to_dict(orient="index")}

## Colour analysis (held-out set, 500 pairs)

In [ ]:
from color_analysis import analyze, build_test_set, summary_tables, save_tables, _Acc, srgb_to_lab, _to01

A_dev, B_dev, gray_dev = build_test_set(clean_dir=CLEAN_DIR, n=500, grayscale_prob=0.5, seed=0)
torch.manual_seed(0)
stats = analyze(obj, {"Ours": ours}, A_dev, B_dev, gray_dev, rf_steps=(None,))
extra = {}
if run2:
    extra["Ours, run 2"] = (run2[0], None)
if flow:
    extra["Bridge flow"] = (flow[0], 1)
for name, (m, k) in extra.items():
    torch.manual_seed(0)
    st = analyze(obj, {name: m}, A_dev, B_dev, gray_dev, rf_steps=(k,), include_input=False)
    stats[name] = st[name if k is None else f"{name} ({k}-step)"]
acc = _Acc(dev)
with torch.no_grad():
    for i in range(0, len(B_dev), 16):
        b = B_dev[i:i + 16].to(dev)
        o, g = _to01(obj.decode_B(obj.encode_B(b))), _to01(b)
        acc.add(srgb_to_lab(o), srgb_to_lab(g), o, g, gray_dev[i:i + 16].to(dev), 8.0, 5.0)
stats["VAE B ceiling"] = acc.finish()
colour_df, gray_df = summary_tables(stats)
print("COLOUR images"); display(colour_df)
print("GRAY images (any colour is false colour)"); display(gray_df)
save_tables(stats, f"{OUT_DIR}/colour")
summary["colour"] = {"colour": colour_df.to_dict(orient="index"), "gray": gray_df.to_dict(orient="index")}

## Real old photographs (no-reference metrics; raw outputs)

In [ ]:
import pyiqa
from torchvision.transforms.functional import to_tensor


def restore(img, net):
    obj.mapping, obj.uses_mask = net, False
    try:
        with torch.no_grad():
            return obj.restore(img, keep_size=True, normalize=False)
    finally:
        obj.mapping, obj.uses_mask = ms_mapping, ms_uses_mask


def fit(img):
    if MAX_SIDE and max(img.size) > MAX_SIDE:
        s = MAX_SIDE / max(img.size)
        img = img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)
    return img


iqa = {n: pyiqa.create_metric(n, device=dev) for n in ("musiq", "clipiqa", "niqe")}
os.makedirs(f"{OUT_DIR}/real", exist_ok=True)
real, rows = {}, []
for p in sorted(f for f in glob.glob(os.path.join(REAL_DIR, "*")) if f.lower().endswith(IMG_EXT)):
    name = os.path.splitext(os.path.basename(p))[0]
    img = fit(Image.open(p).convert("RGB"))
    outs = {"input": img, "BOPBTL": restore(img, ms_net), "Ours": restore(img, ours)}
    for k, im in outs.items():
        im.save(f"{OUT_DIR}/real/{name}_{k}.png")
        with torch.no_grad():
            rows.append({"photo": name, "method": k, **{m: float(f(to_tensor(im).unsqueeze(0).to(dev))) for m, f in iqa.items()}})
    real[name] = outs
nr = pd.DataFrame(rows)
nr.to_csv(f"{OUT_DIR}/real_photos.csv", index=False)
means = nr.groupby("method")[list(iqa)].mean().loc[["input", "BOPBTL", "Ours"]]
display(means.round(4))
summary["real_photos"] = {"n_photos": len(real), "means": means.round(4).to_dict(orient="index")}

## Figures

In [ ]:
import random
import matplotlib.pyplot as plt
from benchmark import BenchPairs

ds = BenchPairs(BENCH_DIR)
r = random.Random(0)
idx = r.sample([i for i, row in enumerate(ds.rows) if row["gray"] == "1"], 2) + \
      r.sample([i for i, row in enumerate(ds.rows) if row["gray"] == "0"], 2)
to_pil = lambda t: Image.fromarray(((t.clamp(-1, 1) + 1) * 127.5).round().byte().permute(1, 2, 0).numpy())
fig_rows = []
for i in idx:
    A_, B_, _ = ds[i]
    a = to_pil(A_)
    fig_rows.append((f"COCO #{ds.rows[i]['idx']}", [a, restore(a, ms_net), restore(a, ours), to_pil(B_)]))
for name in sorted(real)[:3]:
    fig_rows.append((f"real: {name}", [real[name]["input"], real[name]["BOPBTL"], real[name]["Ours"], None]))
fig, axes = plt.subplots(len(fig_rows), 4, figsize=(12, 3.1 * len(fig_rows)), squeeze=False)
for i_, (label, ims) in enumerate(fig_rows):
    for c, im in enumerate(ims):
        axes[i_, c].axis("off")
        if im is not None:
            axes[i_, c].imshow(im)
        if i_ == 0:
            axes[i_, c].set_title(["Input", "BOPBTL", "Ours", "Clean target"][c])
    axes[i_, 0].text(-0.04, 0.5, label, transform=axes[i_, 0].transAxes, rotation=90, va="center", ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/qualitative.pdf", bbox_inches="tight", dpi=200)
plt.show()

hist_path = next((os.path.join(os.path.dirname(p), "history.csv") for p in (OURS, OURS_RUN2)
                  if p and os.path.exists(os.path.join(os.path.dirname(p), "history.csv"))), None)
if hist_path:
    h = pd.read_csv(hist_path)
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
    ax[0].plot(h["step"], h["val_latent_mse"]); ax[0].set_title("validation latent MSE")
    for n_, lbl in (("gray", "gray targets"), ("color", "colour targets")):
        if f"val_{n_}_l1" in h:
            ax[1].plot(h["step"], h[f"val_{n_}_l1"], label=lbl)
    ax[1].set_title("validation image L1"); ax[1].legend()
    ax[2].plot(h["step"], h["train_d"]); ax[2].set_title("discriminator loss (train)")
    for a_ in ax:
        a_.axvline(2000, ls="--", c="gray", lw=0.8); a_.set_xlabel("step")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/training_curve.pdf", bbox_inches="tight"); plt.show()

json.dump(summary, open(f"{OUT_DIR}/summary.json", "w"), indent=2, default=str)
print("saved", f"{OUT_DIR}/summary.json")